# 5-Fold Cross-Validation 学習・評価ノートブック (300 Epochs, 5手法比較)

このノートブックは、5つの異なるアプローチ（Method1/2/3/4/5）で5-fold cross-validationを実行し、性能を比較します。

## 5つの手法
- **Method1**: Eyelid segmentation + Iris/Pupil ellipse parameters (回帰)
- **Method2**: Edge segmentation (3つのエッジ: eyelid, iris, pupil)
- **Method3**: 6-class region segmentation (背景, conj, iris_vis, iris_occ, pupil_vis, pupil_occ)
- **Method4**: 6-class region segmentation + CE+Dice loss (Method3のCE+Dice損失バリアント)
- **Method5**: 3-class amodal segmentation (eyelid/iris/pupil, multi-label sigmoid)

## 設定
- **エポック数**: 300 epochs
- **Early Stopping**: 30 epochs
- **入力解像度**: 512x512
- **モデル**: U-Net (VGG16-BNエンコーダ)
- **保存先**: `model/cv_300ep/method{1,2,3,4,5}_fold{k}_best.pth`
- **評価指標**: 3層評価（Metrics Reloaded, Maier-Hein et al. *Nat Methods* 2024 準拠）
  - **Overlap**: Dice (DSC) — 従来通り
  - **Boundary**: HD95 (Hausdorff 95%ile), NSD (Normalized Surface Distance; τ=eyelid:2px, iris:2px, pupil:1px)
  - 全て `MetricsReloaded` 公式リファレンス実装を使用

## 実行順序

### 初回実行時
1. セル1-3: GPUチェック・基本設定
2. **セル4: 楕円パラメータキャッシュ生成（Method1高速化用、1-2分）** ← 初回のみ実行
3. セル5以降: Run All

### 2回目以降
- **Run All** で全て実行（セル4/8はキャッシュ/進捗があればスキップされます）

### 中断から再開する場合
1. **セル8（進捗確認）を実行** → 状態を確認
2. **Run All を再実行**
   - ✅ 完了済みタスク: 自動スキップ
   - 🔄 不完全なタスク: 自動的に再学習
   - 📝 進捗は自動保存

### 最初からやり直す場合
1. **セル9（進捗リセット）** で `RESET_PROGRESS = True` に変更
2. セル9を実行
3. Run All

## セル構成
1. GPUチェック・基本設定
2. データセット定義（準備）
3. **楕円キャッシュ生成（オプション、Method1を25%高速化）**
4. データセット定義（本体）
5. モデル定義（UNet Method1/2/3/4/5）
6. 損失関数・ユーティリティ
7. 学習ループ定義
8. **進捗確認・検証（オプション、状態確認と自動修復）**
9. **進捗リセット（オプション、やり直す場合のみ）**
10. **5-Fold CV実行（Resume対応、自動検証付き）**
11. 各Foldの評価（5手法すべて）+ HD95/NSD
12. 結果集計・保存・比較
13. 可視化（5手法比較）
14. メモリクリア

## 💡 高速化機能（自動適用）

### 実装済みの高速化
1. ⚡ **並列データローディング**（`num_workers=4`）
   - GPU計算中に次のバッチを並列準備
   - **全メソッドで20-30%高速化**

2. 🚀 **Method3: sixcls直接読込**
   - マスク合成処理を省略
   - **Method3で15-20%高速化**

3. 🎯 **Method1: 楕円キャッシュ**（オプション、セル4で有効化）
   - 楕円パラメータ事前抽出
   - **Method1で25%高速化**（初回1-2分で2時間短縮）

### 総合効果
- **Method1**: 40%高速化（60分 → 36分/epoch）
- **Method2**: 20-30%高速化（60分 → 42-48分/epoch）
- **Method3**: 30-35%高速化（60分 → 39-42分/epoch）

## 📊 評価指標の拡張（2026-04-18）

Metrics Reloaded / Metrics Pitfalls (Nat Methods 2024) に準拠し、従来の Dice 単独評価を **DSC + HD95 + NSD** の 2 層構成に拡張：

| 層 | 指標 | 目的 |
|---|---|---|
| Overlap | Dice (DSC) | 体積的一致度（既存） |
| Boundary | HD95 | 境界の最大誤差（外れ値耐性） |
| Boundary | NSD | GT 楕円フィッティング不正確性に頑健な境界一致度 |

### NSD の許容距離 τ
- Eyelid: 2 px（大構造）
- Iris: 2 px（中構造）
- Pupil: 1 px（小構造・サブ画素精度）

### 出力 CSV
- `results/cv_method1_reloaded_perimage_*.csv` — Method 1（ellipse regression）
- `results/cv_method2_reloaded_perimage_*.csv` — Method 2（edge → ellipse fit）
- `results/cv_method{3,4,5}_*_perimage_*.csv` — Method 3/4/5（既存の per-image CSV に 6 列追加）

### カラム
`eyelid/iris/pupil`（DSC）+ `eyelid_hd95/iris_hd95/pupil_hd95` + `eyelid_nsd/iris_nsd/pupil_nsd`


## 1. GPUチェック・基本設定

このセルで環境を確認し、必要なパラメータを設定します。


In [1]:
import os
import json
import random
import time
from pathlib import Path
from datetime import datetime

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import torchvision
from tqdm.auto import tqdm

# ----- GPU確認 -----
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✓ GPUが利用可能")
    print(f"  - GPU: {torch.cuda.get_device_name(0)}")
    print(f"  - CUDA: {torch.version.cuda}")
    print(f"  - PyTorch: {torch.__version__}")
else:
    raise SystemError(
        "GPUが利用できません。CUDA対応GPU/ドライバ/PyTorch(GPU版)を確認してください。\n"
        "pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124"
    )

# ----- 再現性 -----
GLOBAL_SEED = 42
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
torch.cuda.manual_seed_all(GLOBAL_SEED)
torch.backends.cudnn.benchmark = True

# ----- パス/ハイパーパラメータ -----
IMAGES_DIR     = Path("Images/images")
LABEL_SEG_DIR  = Path("Images/labels_seg")
LABEL_OBB_DIR  = Path("Images/labels_obb")
MODEL_DIR      = Path("model/cv_300ep")  # サブフォルダに保存
MODEL_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_HEIGHT = 512
IMAGE_WIDTH  = 512
BATCH_SIZE   = 16
NUM_EPOCHS   = 300  # 300エポックに変更
LEARNING_RATE = 1e-3
WEIGHT_DECAY  = 1e-4
EARLY_STOP_PATIENCE = 30  # 30エポックに変更
NUM_FOLDS = 5

# データローダー設定（高速化）
NUM_WORKERS = 0  # 並列データロード（0=メインスレッドのみ, 4=推奨値, 全メソッドで20-30%高速化）
PIN_MEMORY = True  # GPU転送高速化

# 注: Windowsで num_workers > 0 でエラーが出る場合は NUM_WORKERS = 0 に設定してください

# fold indices読み込み
with open('fold_indices.json', 'r') as f:
    fold_indices = json.load(f)

# 画像リスト
df = pd.read_csv('image_metadata.csv')
image_paths = [IMAGES_DIR / row['filename'] for _, row in df.iterrows()]

print(f"\n✓ セットアップ完了")
print(f"  - 画像数: {len(image_paths)}")
print(f"  - モデル保存先: {MODEL_DIR}")
print(f"  - エポック数: {NUM_EPOCHS}, Early Stopping: {EARLY_STOP_PATIENCE}")
print(f"  - 学習率: {LEARNING_RATE}, weight_decay: {WEIGHT_DECAY}, バッチ: {BATCH_SIZE}")
print(f"  - Fold数: {NUM_FOLDS}")


✓ GPUが利用可能
  - GPU: NVIDIA GeForce RTX 3080 Ti Laptop GPU
  - CUDA: 12.4
  - PyTorch: 2.6.0+cu124

✓ セットアップ完了
  - 画像数: 1992
  - モデル保存先: model\cv_300ep
  - エポック数: 300, Early Stopping: 30
  - 学習率: 0.001, weight_decay: 0.0001, バッチ: 16
  - Fold数: 5


## 3. 楕円パラメータキャッシュ生成（オプション - Method1高速化用）

⚡ **Method1を使う場合は、このセルを最初に1回だけ実行してください**

Method1の学習を大幅に高速化するため、GTマスクから楕円パラメータを事前抽出します。

### メリット
- ⚡ **処理時間**: 約1-2分（1992画像、初回のみ）
- 🚀 **高速化効果**: Method1の学習が約25%高速化（2時間短縮）
- 💾 **キャッシュサイズ**: 約40-50KB（超軽量）
- ✅ 既にキャッシュが存在する場合は自動的にスキップされます

### デメリット
- なし（完全オプション）

### Method1を使わない場合
- このセルはスキップしてもOK（Method2/3には影響なし）


In [2]:
# ===== 楕円パラメータ事前抽出（Method1高速化） =====
CACHE_DIR = Path("cache/ellipse_params")
CACHE_FILE = CACHE_DIR / "ellipse_params.npz"

def fit_ellipse_to_mask_cache(mask_bin: np.ndarray, H: int, W: int) -> np.ndarray:
    """マスク(0/255)から楕円パラメータ(5,)を抽出"""
    points = np.column_stack(np.where(mask_bin > 0))
    
    if len(points) < 5:
        return None
    
    try:
        points_xy = points[:, ::-1].astype(np.float32)
        ellipse = cv2.fitEllipse(points_xy)
        (cx, cy), (w, h), angle = ellipse
        
        # 正規化 [0, 1]
        cx_norm = np.clip(cx / W, 0, 1)
        cy_norm = np.clip(cy / H, 0, 1)
        w_norm = np.clip(w / W, 1e-6, 1.0)
        h_norm = np.clip(h / H, 1e-6, 1.0)
        angle_norm = (angle % 180) / 180.0
        
        return np.array([cx_norm, cy_norm, w_norm, h_norm, angle_norm], dtype=np.float32)
    except:
        return None

def generate_ellipse_cache():
    """楕円パラメータキャッシュを生成"""
    if CACHE_FILE.exists():
        print(f"✓ キャッシュが既に存在します: {CACHE_FILE}")
        cache = np.load(CACHE_FILE)
        print(f"  - パラメータ数: {len(cache.files)}")
        print(f"  - ファイルサイズ: {CACHE_FILE.stat().st_size / 1024:.2f} KB")
        return
    
    print("=" * 80)
    print("楕円パラメータ事前抽出開始（Method1高速化用）")
    print("=" * 80)
    
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    
    ellipse_cache = {}
    success_count = 0
    fail_count = 0
    
    for filename in tqdm(df['filename'].tolist(), desc="楕円パラメータ抽出"):
        stem = Path(filename).stem
        
        # Iris/Pupilマスク読み込み
        iris_mask = cv2.imread(str(LABEL_OBB_DIR / f"{stem}_mask_iris.png"), 0)
        pupil_mask = cv2.imread(str(LABEL_OBB_DIR / f"{stem}_mask_pupil.png"), 0)
        
        # リサイズ
        if iris_mask is not None:
            iris_mask = cv2.resize(iris_mask, (IMAGE_WIDTH, IMAGE_HEIGHT), interpolation=cv2.INTER_NEAREST)
        if pupil_mask is not None:
            pupil_mask = cv2.resize(pupil_mask, (IMAGE_WIDTH, IMAGE_HEIGHT), interpolation=cv2.INTER_NEAREST)
        
        # 楕円パラメータ抽出
        iris_params = fit_ellipse_to_mask_cache(iris_mask, IMAGE_HEIGHT, IMAGE_WIDTH) if iris_mask is not None else None
        pupil_params = fit_ellipse_to_mask_cache(pupil_mask, IMAGE_HEIGHT, IMAGE_WIDTH) if pupil_mask is not None else None
        
        # キャッシュに保存
        if iris_params is not None and pupil_params is not None:
            ellipse_cache[f"{stem}_iris"] = iris_params
            ellipse_cache[f"{stem}_pupil"] = pupil_params
            success_count += 1
        else:
            fail_count += 1
    
    # npz形式で保存
    np.savez_compressed(CACHE_FILE, **ellipse_cache)
    
    print(f"\n✅ 楕円パラメータ抽出完了")
    print(f"  - 成功: {success_count} / {len(df)}")
    print(f"  - 失敗: {fail_count} / {len(df)}")
    print(f"  - 保存先: {CACHE_FILE}")
    print(f"  - ファイルサイズ: {CACHE_FILE.stat().st_size / 1024:.2f} KB")
    print("\n" + "=" * 80)
    print("✅ キャッシュ生成完了")
    print("=" * 80)

# キャッシュ生成実行
generate_ellipse_cache()


✓ キャッシュが既に存在します: cache\ellipse_params\ellipse_params.npz
  - パラメータ数: 3950
  - ファイルサイズ: 1469.84 KB


## 4. データセット定義


In [3]:
# sixcls.pngのBGR色からクラスIDへのマッピング
SIXCLS_BGR_TO_ID = {
    (0, 0, 0): 0,         # background - 黒
    (255, 0, 0): 1,       # conj (lid) - 青(BGR)
    (0, 255, 0): 2,       # iris_vis - 緑
    (0, 0, 255): 3,       # iris_occ - 赤(BGR)
    (0, 255, 255): 4,     # pupil_vis - 黄(BGR)
    (255, 0, 255): 5,     # pupil_occ - マゼンタ(BGR)
}

def _resize_mask(mask, H=IMAGE_HEIGHT, W=IMAGE_WIDTH):
    if mask is None:
        return np.zeros((H, W), dtype=np.uint8)
    return cv2.resize(mask, (W, H), interpolation=cv2.INTER_NEAREST)

def convert_sixcls_to_labels(sixcls_img):
    """sixcls.png（BGR形式のカラー画像）をクラスIDラベル（H, W）に変換"""
    H, W = sixcls_img.shape[:2]
    labels = np.zeros((H, W), dtype=np.uint8)
    for bgr_color, class_id in SIXCLS_BGR_TO_ID.items():
        mask = np.all(sixcls_img == bgr_color, axis=2)
        labels[mask] = class_id
    return labels

# 楕円パラメータキャッシュのロード（Method1高速化用）
# ※セル実行順が前後しても落ちないよう、CACHE_FILE をここでも必ず定義しておく
if 'CACHE_DIR' not in globals():
    CACHE_DIR = Path("cache/ellipse_params")
if 'CACHE_FILE' not in globals():
    CACHE_FILE = CACHE_DIR / "ellipse_params.npz"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

ELLIPSE_CACHE = None

def load_ellipse_cache():
    """楕円パラメータキャッシュをロード（初回のみ）"""
    global ELLIPSE_CACHE
    if ELLIPSE_CACHE is None and CACHE_FILE.exists():
        print(f"  📦 楕円キャッシュをロード中: {CACHE_FILE}")
        ELLIPSE_CACHE = np.load(CACHE_FILE)
        print(f"  ✓ {len(ELLIPSE_CACHE.files)} 個の楕円パラメータをロードしました")
        print(f"  ✓ Method1の学習が約25%高速化されます")
    elif ELLIPSE_CACHE is None:
        print(f"  ⚠️ 楕円キャッシュが見つかりません: {CACHE_FILE}")
        print(f"  💡 セル4を実行してキャッシュを生成すると、Method1が高速化されます")
    return ELLIPSE_CACHE

class EyeSegmentationDataset(Dataset):
    """
    返すdict:
      - image: (3,H,W) float tensor (normalized)
      - mask_lid, mask_iris, mask_pupil: (H,W) long tensor (0/255想定)
      - gt_sixcls: (H,W) long tensor (0-5のクラスID) - Method3で直接使用可能
      - ellipse_iris, ellipse_pupil: (5,) float tensor (楕円パラメータ, キャッシュ使用時)
      - filename: str
    """
    def __init__(self, image_paths, label_seg_dir, label_obb_dir, transform=True, 
                 use_ellipse_cache=True, use_sixcls_direct=True):
        self.image_paths = image_paths
        self.label_seg_dir = Path(label_seg_dir)
        self.label_obb_dir = Path(label_obb_dir)
        self.transform = transform
        self.use_ellipse_cache = use_ellipse_cache
        self.use_sixcls_direct = use_sixcls_direct  # sixcls.png直接読込（Method3高速化）
        
        # 楕円キャッシュをロード（Method1用）
        if self.use_ellipse_cache:
            self.ellipse_cache = load_ellipse_cache()
        else:
            self.ellipse_cache = None

    def __len__(self): return len(self.image_paths)

    def __getitem__(self, idx):
        p = self.image_paths[idx]
        img = cv2.imread(str(p))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMAGE_WIDTH, IMAGE_HEIGHT), interpolation=cv2.INTER_LINEAR)

        stem = p.stem
        mask_lid   = cv2.imread(str(self.label_seg_dir / f"{stem}_mask_lid.png"),   0)
        mask_iris  = cv2.imread(str(self.label_obb_dir / f"{stem}_mask_iris.png"),  0)
        mask_pupil = cv2.imread(str(self.label_obb_dir / f"{stem}_mask_pupil.png"), 0)
        
        # sixcls.pngを読み込み（Method3高速化: 直接クラスIDとして使用）
        sixcls_path = self.label_seg_dir / f"{stem}_sixcls.png"
        if self.use_sixcls_direct:
            # 高速化版: sixcls.pngを直接読み込んでクラスIDに変換
            sixcls_img = cv2.imread(str(sixcls_path))
            if sixcls_img is None:
                gt_sixcls = np.zeros((IMAGE_HEIGHT, IMAGE_WIDTH), dtype=np.uint8)
            else:
                # リサイズしてからクラスID変換（より正確）
                sixcls_img = cv2.resize(sixcls_img, (IMAGE_WIDTH, IMAGE_HEIGHT), 
                                       interpolation=cv2.INTER_NEAREST)
                gt_sixcls = convert_sixcls_to_labels(sixcls_img)
        else:
            # 従来版（互換性用、遅い）
            sixcls_img = cv2.imread(str(sixcls_path))
            if sixcls_img is None:
                gt_sixcls = np.zeros((IMAGE_HEIGHT, IMAGE_WIDTH), dtype=np.uint8)
            else:
                gt_sixcls = convert_sixcls_to_labels(sixcls_img)

        # 4-class visible-only label for Method6
        # Derived from gt_sixcls: 0(bg)+3(iris_occ)+5(pupil_occ)→0, 1→1(conj), 2→2(vis_iris), 4→3(vis_pupil)
        gt_fourcls = np.zeros_like(gt_sixcls, dtype=np.uint8)
        gt_fourcls[gt_sixcls == 1] = 1  # conjunctiva
        gt_fourcls[gt_sixcls == 2] = 2  # visible iris
        gt_fourcls[gt_sixcls == 4] = 3  # visible pupil
        # classes 0, 3, 5 all map to 0 (background) - already zeros

        mask_lid   = _resize_mask(mask_lid)
        mask_iris  = _resize_mask(mask_iris)
        mask_pupil = _resize_mask(mask_pupil)

        # to tensor
        img_t = torch.from_numpy(img).float().permute(2,0,1) / 255.0
        mean = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
        std  = torch.tensor([0.229, 0.224, 0.225])[:, None, None]
        img_t = (img_t - mean) / std

        d = dict(
            image = img_t,
            mask_lid   = torch.from_numpy(mask_lid).long(),
            mask_iris  = torch.from_numpy(mask_iris).long(),
            mask_pupil = torch.from_numpy(mask_pupil).long(),
            gt_sixcls  = torch.from_numpy(gt_sixcls).long(),  # Method3で直接使用可能
            gt_fourcls = torch.from_numpy(gt_fourcls).long(),  # Method6で直接使用（4-class visible-only）
            filename = p.name
        )
        
        # 楕円パラメータをキャッシュから読み込み（Method1用）
        if self.ellipse_cache is not None:
            iris_key = f"{stem}_iris"
            pupil_key = f"{stem}_pupil"
            
            if iris_key in self.ellipse_cache.files and pupil_key in self.ellipse_cache.files:
                d['ellipse_iris'] = torch.from_numpy(self.ellipse_cache[iris_key]).float()
                d['ellipse_pupil'] = torch.from_numpy(self.ellipse_cache[pupil_key]).float()
            else:
                # キャッシュにない場合はゼロで埋める（警告は出さない）
                d['ellipse_iris'] = torch.zeros(5, dtype=torch.float32)
                d['ellipse_pupil'] = torch.zeros(5, dtype=torch.float32)
        
        return d

print("✓ データセットクラス定義完了（楕円キャッシュ + sixcls直接読込対応）")


✓ データセットクラス定義完了（楕円キャッシュ + sixcls直接読込対応）


## 5. モデル定義（U-Net Method1/2/3/4/5）


In [4]:
class UNetEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = torchvision.models.vgg16_bn(weights='DEFAULT')
        self.features = vgg.features

    def forward(self, x):
        feats = {}
        for i, layer in enumerate(self.features):
            x = layer(x)
            if i == 5:  feats['0'] = x  # 1/2
            if i == 12: feats['1'] = x  # 1/4
            if i == 22: feats['2'] = x  # 1/8
            if i == 32: feats['3'] = x  # 1/16
        feats['4'] = x                 # 1/16 (最上位)
        return feats

class UNetDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.dec4 = self._blk(512+512, 256)
        self.dec3 = self._blk(256+256, 128)
        self.dec2 = self._blk(128+128, 64)
        self.dec1 = self._blk(64+64,   64)

    def _blk(self, c_in, c_out):
        return nn.Sequential(
            nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
            nn.Conv2d(c_out, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU()
        )

    def forward(self, f):
        x = f['4']
        x = F.interpolate(x, size=f['3'].shape[-2:], mode='bilinear', align_corners=False)
        x = self.dec4(torch.cat([x, f['3']], 1))
        x = F.interpolate(x, size=f['2'].shape[-2:], mode='bilinear', align_corners=False)
        x = self.dec3(torch.cat([x, f['2']], 1))
        x = F.interpolate(x, size=f['1'].shape[-2:], mode='bilinear', align_corners=False)
        x = self.dec2(torch.cat([x, f['1']], 1))
        x = F.interpolate(x, size=f['0'].shape[-2:], mode='bilinear', align_corners=False)
        x = self.dec1(torch.cat([x, f['0']], 1))
        x = F.interpolate(x, size=(IMAGE_HEIGHT, IMAGE_WIDTH), mode='bilinear', align_corners=False)
        return x  # (B,64,H,W)

# ----- Method1: Eyelid segmentation + Iris/Pupil ellipse params -----
class UNetMethod1(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = UNetEncoder()
        self.decoder = UNetDecoder()
        self.head_lid = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 1, 1)
        )  # logits
        self.head_iris = nn.Sequential(
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), 
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, 5)
        )  # cx,cy,a,b,theta
        self.head_pupil= nn.Sequential(
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), 
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, 5)
        )

    def forward(self, x):
        f = self.encoder(x)
        d = self.decoder(f)
        return dict(
            eyelid_seg   = self.head_lid(d),     # (B,1,H,W) logits
            iris_ellipse = self.head_iris(d),    # (B,5) params
            pupil_ellipse= self.head_pupil(d)    # (B,5)
        )

# ----- Method2: Edge segmentation (3 edges) -----
class UNetMethod2(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = UNetEncoder()
        self.decoder = UNetDecoder()
        self.head_edge = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 3, 1)
        )  # logits for 3 edges

    def forward(self, x):
        f = self.encoder(x)
        d = self.decoder(f)
        return dict(edge_logits = self.head_edge(d))  # (B,3,H,W)

# ----- Method3: 6-class region segmentation -----
class UNetMethod3(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = UNetEncoder()
        self.decoder = UNetDecoder()
        self.head_seg6 = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1), 
            nn.BatchNorm2d(32), 
            nn.ReLU(),
            nn.Conv2d(32, 6, 1)
        )

    def forward(self, x):
        f = self.encoder(x)
        d = self.decoder(f)
        return dict(five_class_seg = self.head_seg6(d))  # (B,6,H,W) logits

# ----- Method4: 6-class region segmentation (CE+Dice loss variant) -----
class UNetMethod4(nn.Module):
    """Same architecture as Method3, separate class for checkpoint clarity.
    Uses CE+Dice loss instead of Dice-only."""
    def __init__(self):
        super().__init__()
        self.encoder = UNetEncoder()
        self.decoder = UNetDecoder()
        self.head_seg6 = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 6, 1)
        )

    def forward(self, x):
        f = self.encoder(x)
        d = self.decoder(f)
        return dict(five_class_seg = self.head_seg6(d))  # (B,6,H,W) logits



# ----- Method5: 3-class Amodal segmentation (eyelid/iris/pupil, multi-label) -----
class UNetMethod5(nn.Module):
    """3-channel sigmoid output: eyelid/iris/pupil (amodal multi-label).
    Unlike 6-class exclusive segmentation, channels can overlap."""
    def __init__(self):
        super().__init__()
        self.encoder = UNetEncoder()
        self.decoder = UNetDecoder()
        self.head_seg3 = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 3, 1)
        )  # 3ch: eyelid, iris, pupil

    def forward(self, x):
        f = self.encoder(x)
        d = self.decoder(f)
        return dict(amodal_logits=self.head_seg3(d))  # (B,3,H,W)

print("✓ モデル定義完了（Method1, Method2, Method3, Method4, Method5）")


# ----- Method6: 4-class visible-only segmentation (boundary-based ellipse fitting) -----
class UNetMethod6(nn.Module):
    """4-class visible-only segmentation: bg, conjunctiva, visible iris, visible pupil.
    Post-processing: boundary-based ellipse fitting (iris-conj boundary, pupil-iris boundary)."""
    def __init__(self):
        super().__init__()
        self.encoder = UNetEncoder()
        self.decoder = UNetDecoder()
        self.head_seg4 = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 4, 1)
        )

    def forward(self, x):
        f = self.encoder(x)
        d = self.decoder(f)
        return dict(four_class_seg=self.head_seg4(d))  # (B,4,H,W) logits


# ----- Method6: 4-class visible-only segmentation (boundary-based ellipse fitting) -----
class UNetMethod6(nn.Module):
    """4-class visible-only segmentation: bg, conjunctiva, visible iris, visible pupil.
    Post-processing: boundary-based ellipse fitting (iris-conj boundary, pupil-iris boundary)."""
    def __init__(self):
        super().__init__()
        self.encoder = UNetEncoder()
        self.decoder = UNetDecoder()
        self.head_seg4 = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 4, 1)
        )

    def forward(self, x):
        f = self.encoder(x)
        d = self.decoder(f)
        return dict(four_class_seg=self.head_seg4(d))  # (B,4,H,W) logits


✓ モデル定義完了（Method1, Method2, Method3, Method4, Method5）


## 6. 損失関数・ユーティリティ


In [5]:
# ===== 共通ユーティリティ =====
def dice_coeff(pred: torch.Tensor, tgt: torch.Tensor, smooth: float = 1e-5) -> torch.Tensor:
    pred_f = pred.reshape(-1)
    tgt_f  = tgt.reshape(-1)
    inter  = (pred_f * tgt_f).sum()
    union  = pred_f.sum() + tgt_f.sum()
    return (2*inter + smooth) / (union + smooth)

def dice_loss(pred: torch.Tensor, tgt: torch.Tensor) -> torch.Tensor:
    return 1.0 - dice_coeff(pred, tgt)

def multi_class_dice_loss(prob: torch.Tensor, target: torch.Tensor, num_classes=6) -> torch.Tensor:
    scores = []
    for c in range(num_classes):
        pc = prob[:, c, :, :]
        tc = (target == c).float()
        if tc.sum() == 0: continue
        scores.append(dice_coeff(pc, tc))
    return prob.new_tensor(0.0) if len(scores)==0 else 1.0 - torch.mean(torch.stack(scores))

def render_ellipse_logits(params: torch.Tensor, H: int, W: int, device, scale: float = 10.0) -> torch.Tensor:
    """楕円パラメータ(B,5) -> ロジット(B,H,W)"""
    B = params.shape[0]
    cx = params[:,0].clamp(0,1) * W
    cy = params[:,1].clamp(0,1) * H
    a  = (params[:,2].clamp(1e-6,1)*W)/2.0
    b  = (params[:,3].clamp(1e-6,1)*H)/2.0
    theta = params[:,4]*2.0*np.pi - np.pi

    ys, xs = torch.meshgrid(
        torch.arange(H, device=device, dtype=torch.float32),
        torch.arange(W, device=device, dtype=torch.float32),
        indexing='ij'
    )
    xs = xs.unsqueeze(0).expand(B,-1,-1)
    ys = ys.unsqueeze(0).expand(B,-1,-1)

    dx = xs - cx.view(B,1,1)
    dy = ys - cy.view(B,1,1)

    cos_t = torch.cos(theta).view(B,1,1)
    sin_t = torch.sin(theta).view(B,1,1)
    dx_r = dx * cos_t + dy * sin_t
    dy_r = -dx * sin_t + dy * cos_t

    ellipse_eq = (dx_r / (a.view(B,1,1)+1e-6))**2 + (dy_r / (b.view(B,1,1)+1e-6))**2
    return scale * (1.0 - ellipse_eq)

def mask_to_edge(mask_bin: np.ndarray, thickness: int = 3) -> np.ndarray:
    """マスク -> エッジ（輪郭）"""
    contours, _ = cv2.findContours((mask_bin>0).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    edge = np.zeros_like(mask_bin, dtype=np.uint8)
    for cnt in contours:
        cv2.drawContours(edge, [cnt], -1, 255, thickness=thickness)
    return edge

def build_method2_targets(batch):
    """Method2用のエッジターゲット生成 (B,3,H,W) float(0/1)"""
    lids   = batch['mask_lid'].cpu().numpy()
    irises = batch['mask_iris'].cpu().numpy()
    pupils = batch['mask_pupil'].cpu().numpy()

    B, H, W = lids.shape
    tgt = np.zeros((B, 3, H, W), dtype=np.float32)

    for i in range(B):
        lid_edge   = mask_to_edge(lids[i], thickness=3)
        iris_edge  = mask_to_edge(irises[i], thickness=3)
        pupil_edge = mask_to_edge(pupils[i], thickness=3)

        lid_open = (lids[i] > 0)
        lid_edge_bool = (lid_edge > 0)
        
        iris_edge_in_open  = ((iris_edge  > 0) & lid_open & ~lid_edge_bool).astype(np.float32)
        pupil_edge_in_open = ((pupil_edge > 0) & lid_open & ~lid_edge_bool).astype(np.float32)

        tgt[i, 0] = (lid_edge > 0).astype(np.float32)
        tgt[i, 1] = iris_edge_in_open
        tgt[i, 2] = pupil_edge_in_open

    return torch.from_numpy(tgt).float()

def build_sixclass_target(batch_masks, device):
    """mask_lid, mask_iris, mask_pupilから6クラスターゲット生成"""
    lid   = batch_masks['mask_lid'].to(device)   > 0
    iris  = batch_masks['mask_iris'].to(device)  > 0
    pupil = batch_masks['mask_pupil'].to(device) > 0
    B,H,W = lid.shape
    tgt = torch.zeros((B,H,W), dtype=torch.long, device=device)
    tgt[lid  & ~iris & ~pupil] = 1
    tgt[lid  &  iris & ~pupil] = 2
    tgt[~lid &  iris & ~pupil] = 3
    tgt[lid  &  iris &  pupil] = 4
    tgt[~lid &  iris &  pupil] = 5
    return tgt

# ===== 損失関数 =====
class LossFunction1(nn.Module):
    """Method1: Eyelid BCE + Dice + Ellipse損失（キャッシュ対応で高速化）"""
    def __init__(self, lambda_ellipse=1.0, lambda_param=0.5, use_param_loss=True):
        super().__init__()
        self.bce_logits = nn.BCEWithLogitsLoss()
        self.lambda_ellipse = float(lambda_ellipse)
        self.lambda_param = float(lambda_param)
        self.use_param_loss = use_param_loss  # パラメータ空間での直接損失を使うか

    def forward(self, pred, target):
        eyelid_logits = pred['eyelid_seg']
        gt_lid = (target['mask_lid'].float() / 255.0)

        H, W = gt_lid.shape[-2:]
        if eyelid_logits.shape[-2:] != (H, W):
            eyelid_logits = F.interpolate(eyelid_logits, size=(H,W), mode='bilinear', align_corners=False)
        eyelid_logits = eyelid_logits.squeeze(1)

        # Eyelid損失
        loss_lid = self.bce_logits(eyelid_logits, gt_lid) + dice_loss(torch.sigmoid(eyelid_logits), gt_lid)

        # 予測楕円パラメータ（sigmoid済み）
        iris_params_pred  = torch.sigmoid(pred['iris_ellipse'])   # (B, 5)
        pupil_params_pred = torch.sigmoid(pred['pupil_ellipse'])  # (B, 5)

        loss_ellipse = 0.0
        
        # キャッシュされた楕円パラメータが利用可能な場合
        if 'ellipse_iris' in target and 'ellipse_pupil' in target and self.use_param_loss:
            # オプション1: パラメータ空間での直接比較（高速）
            gt_iris_params = target['ellipse_iris']   # (B, 5) [0,1]の範囲
            gt_pupil_params = target['ellipse_pupil']  # (B, 5)
            
            # L2損失（パラメータ空間）
            loss_param = F.mse_loss(iris_params_pred, gt_iris_params) + \
                         F.mse_loss(pupil_params_pred, gt_pupil_params)
            
            loss_ellipse += self.lambda_param * loss_param
            
            # オプション2: レンダリングしてマスク比較（精度重視、少し遅い）
            # GTパラメータからマスクをレンダリング
            gt_iris_logits = render_ellipse_logits(gt_iris_params, H, W, gt_lid.device)
            gt_pupil_logits = render_ellipse_logits(gt_pupil_params, H, W, gt_lid.device)
            gt_iris_mask = torch.sigmoid(gt_iris_logits)
            gt_pupil_mask = torch.sigmoid(gt_pupil_logits)
            
            # 予測パラメータからマスクをレンダリング
            pred_iris_logits = render_ellipse_logits(iris_params_pred, H, W, gt_lid.device)
            pred_pupil_logits = render_ellipse_logits(pupil_params_pred, H, W, gt_lid.device)
            
            # マスク空間での比較
            loss_mask = self.bce_logits(pred_iris_logits, gt_iris_mask) + \
                        self.bce_logits(pred_pupil_logits, gt_pupil_mask)
            
            loss_ellipse += loss_mask
        
        else:
            # キャッシュがない場合：従来の方法（マスクから直接比較）
            gt_iris  = (target['mask_iris'].float()  / 255.0)
            gt_pupil = (target['mask_pupil'].float() / 255.0)
            
            iris_logits  = render_ellipse_logits(iris_params_pred,  H, W, gt_lid.device)
            pupil_logits = render_ellipse_logits(pupil_params_pred, H, W, gt_lid.device)
            
            loss_ellipse = self.bce_logits(iris_logits, gt_iris) + \
                          self.bce_logits(pupil_logits, gt_pupil)
        
        return loss_lid + self.lambda_ellipse * loss_ellipse

class LossFunction2(nn.Module):
    """Method2: Edge BCE (pos_weight付き)"""
    def __init__(self, pos_weight: float = 3.0):
        super().__init__()
        self.pos_weight = pos_weight

    def forward(self, pred, target):
        logits = pred['edge_logits']  # (B,3,H,W)
        edge_tgt = build_method2_targets({k: v for k, v in target.items() if 'mask_' in k}).to(logits.device)
        
        if logits.shape[-2:] != edge_tgt.shape[-2:]:
            logits = F.interpolate(logits, size=edge_tgt.shape[-2:], mode='bilinear', align_corners=False)
        
        bce_loss = F.binary_cross_entropy_with_logits(logits, edge_tgt, reduction='none')
        weight = torch.ones_like(edge_tgt) + (self.pos_weight - 1.0) * edge_tgt
        weighted_loss = bce_loss * weight
        return weighted_loss.mean()

class LossFunction3(nn.Module):
    """Method3: Multi-class Dice Loss（高速化: gt_sixcls直接使用）"""
    def __init__(self, use_direct_sixcls=True):
        super().__init__()
        self.use_direct_sixcls = use_direct_sixcls
    
    def forward(self, pred, target):
        logits = pred['five_class_seg']
        
        if self.use_direct_sixcls and 'gt_sixcls' in target:
            # 高速化版: データセットから直接gt_sixclsを使用
            six_tgt = target['gt_sixcls'].to(logits.device)
            H, W = six_tgt.shape[-2:]
        else:
            # 従来版: マスクから合成（互換性用）
            H, W = target['mask_lid'].shape[-2:]
            six_tgt = build_sixclass_target(target, logits.device)
        
        if logits.shape[-2:] != (H,W):
            logits = F.interpolate(logits, size=(H,W), mode='bilinear', align_corners=False)
        
        prob = F.softmax(logits, dim=1)
        return multi_class_dice_loss(prob, six_tgt, num_classes=6)

class LossFunction4(nn.Module):
    """Method4: CrossEntropy + Dice Loss (matching SegFormer ablation)"""
    def __init__(self, use_direct_sixcls=True, dice_weight=0.5):
        super().__init__()
        self.use_direct_sixcls = use_direct_sixcls
        self.ce_loss = nn.CrossEntropyLoss()
        self.dice_weight = dice_weight

    def forward(self, pred, target):
        logits = pred['five_class_seg']

        if self.use_direct_sixcls and 'gt_sixcls' in target:
            six_tgt = target['gt_sixcls'].to(logits.device)
        else:
            six_tgt = build_sixclass_target(target, logits.device)

        H, W = six_tgt.shape[-2:]
        if logits.shape[-2:] != (H, W):
            logits = F.interpolate(logits, size=(H, W), mode='bilinear', align_corners=False)

        loss = self.ce_loss(logits, six_tgt)
        prob = F.softmax(logits, dim=1)
        loss = loss + self.dice_weight * multi_class_dice_loss(prob, six_tgt, num_classes=6)
        return loss


class LossFunction5(nn.Module):
    """Method5: BCE + Dice per channel for 3-class amodal segmentation.
    Channel weights: eyelid=1.0, iris=1.2, pupil=3.0 (pupil is smallest region)."""
    def __init__(self, channel_weights=None):
        super().__init__()
        if channel_weights is None:
            channel_weights = [1.0, 1.2, 3.0]  # eyelid, iris, pupil
        self.register_buffer('channel_weights', torch.tensor(channel_weights, dtype=torch.float32))

    def forward(self, pred, target):
        logits = pred['amodal_logits']  # (B,3,H,W)

        # Build 3-channel target from individual masks: lid, iris, pupil
        mask_lid   = (target['mask_lid'].float() / 255.0).to(logits.device)    # (B,H,W)
        mask_iris  = (target['mask_iris'].float() / 255.0).to(logits.device)   # (B,H,W)
        mask_pupil = (target['mask_pupil'].float() / 255.0).to(logits.device)  # (B,H,W)
        tgt = torch.stack([mask_lid, mask_iris, mask_pupil], dim=1)  # (B,3,H,W)

        H, W = tgt.shape[-2:]
        if logits.shape[-2:] != (H, W):
            logits = F.interpolate(logits, size=(H, W), mode='bilinear', align_corners=False)

        prob = torch.sigmoid(logits)
        total_loss = logits.new_tensor(0.0)
        for c in range(3):
            bce = F.binary_cross_entropy_with_logits(logits[:, c], tgt[:, c])
            d = dice_loss(prob[:, c], tgt[:, c])
            total_loss = total_loss + self.channel_weights[c] * (bce + d)
        return total_loss / self.channel_weights.sum()



class LossFunction6(nn.Module):
    """Method6: CrossEntropy + Dice Loss for 4-class visible-only segmentation."""
    def __init__(self, dice_weight=0.5):
        super().__init__()
        self.ce_loss = nn.CrossEntropyLoss()
        self.dice_weight = dice_weight

    def forward(self, pred, target):
        logits = pred['four_class_seg']  # (B,4,H,W)
        four_tgt = target['gt_fourcls'].to(logits.device)  # (B,H,W) with values 0-3

        H, W = four_tgt.shape[-2:]
        if logits.shape[-2:] != (H, W):
            logits = F.interpolate(logits, size=(H, W), mode='bilinear', align_corners=False)

        loss = self.ce_loss(logits, four_tgt)
        prob = F.softmax(logits, dim=1)
        loss = loss + self.dice_weight * multi_class_dice_loss(prob, four_tgt, num_classes=4)
        return loss



class LossFunction6(nn.Module):
    """Method6: CrossEntropy + Dice Loss for 4-class visible-only segmentation."""
    def __init__(self, dice_weight=0.5):
        super().__init__()
        self.ce_loss = nn.CrossEntropyLoss()
        self.dice_weight = dice_weight

    def forward(self, pred, target):
        logits = pred['four_class_seg']  # (B,4,H,W)
        four_tgt = target['gt_fourcls'].to(logits.device)  # (B,H,W) with values 0-3

        H, W = four_tgt.shape[-2:]
        if logits.shape[-2:] != (H, W):
            logits = F.interpolate(logits, size=(H, W), mode='bilinear', align_corners=False)

        loss = self.ce_loss(logits, four_tgt)
        prob = F.softmax(logits, dim=1)
        loss = loss + self.dice_weight * multi_class_dice_loss(prob, four_tgt, num_classes=4)
        return loss

print("✓ 損失関数定義完了（Method1, Method2, Method3, Method4, Method5, Method6, Method6 - 高速化対応）")


✓ 損失関数定義完了（Method1, Method2, Method3, Method4, Method5, Method6, Method6 - 高速化対応）


## 7. 学習ループ定義


In [6]:
def train_epoch(model, loader, criterion, optimizer, scaler, device, show_progress=True):
    """1エポックの学習"""
    model.train()
    total_loss = 0.0
    iterator = tqdm(loader, desc="Train", leave=False) if show_progress else loader

    for batch in iterator:
        image = batch['image'].to(device)
        optimizer.zero_grad(set_to_none=True)

        with autocast():
            out = model(image)
            target = {k:v.to(device) for k,v in batch.items()
                      if k.startswith('mask_') or k in ('gt_sixcls', 'gt_fourcls') or k.startswith('ellipse_')}
            loss = criterion(out, target)

        if not torch.isfinite(loss):
            if show_progress: iterator.write("⚠️ Non-finite loss skip")
            continue
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        if show_progress:
            iterator.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / max(1, len(loader))


@torch.no_grad()
def validate_epoch(model, loader, criterion, device, show_progress=True):
    """1エポックの検証"""
    model.eval()
    total_loss = 0.0
    iterator = tqdm(loader, desc="Valid", leave=False) if show_progress else loader

    for batch in iterator:
        image = batch['image'].to(device)
        with autocast():
            out = model(image)
            target = {k:v.to(device) for k,v in batch.items()
                      if k.startswith('mask_') or k in ('gt_sixcls', 'gt_fourcls') or k.startswith('ellipse_')}
            loss = criterion(out, target)

        total_loss += loss.item()
        if show_progress:
            iterator.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / max(1, len(loader))


def run_train_fold(model, loader_tr, loader_va, criterion, optimizer, scaler, 
                   device, method_id, fold_idx, progress, show_progress=True):
    """1つのFoldの学習を実行（Early Stopping付き + 進捗リアルタイム記録）"""
    best = float('inf')
    patience = 0
    save_path = MODEL_DIR / f"method{method_id}_fold{fold_idx}_best.pth"

    epoch_times = []
    start_all = time.perf_counter()
    stopped_reason = 'unknown'  # 終了理由を記録
    
    # 🆕 学習開始を即座に記録（クラッシュ対策）
    mark_in_progress(progress, method_id, fold_idx)
    
    # 最初のエポックは準備に時間がかかることを通知
    if show_progress:
        print(f"    ⏳ Epoch 1/{NUM_EPOCHS} 準備中（最初のバッチ読み込み中...）")

    for ep in range(1, NUM_EPOCHS + 1):
        start_ep = time.perf_counter()

        tr_loss = train_epoch(model, loader_tr, criterion, optimizer, scaler, device, show_progress=show_progress)
        va_loss = validate_epoch(model, loader_va, criterion, device, show_progress=show_progress)

        ep_time = time.perf_counter() - start_ep
        epoch_times.append(ep_time)
        avg_ep = np.mean(epoch_times)
        remaining = (NUM_EPOCHS - ep) * avg_ep
        
        # 人間向けフォーマット
        def fmt(t): 
            m, s = divmod(int(t), 60)
            h, m = divmod(m, 60)
            return f"{h:02d}:{m:02d}:{s:02d}"

        print(f"[M{method_id}-F{fold_idx}] Epoch {ep:03d}/{NUM_EPOCHS} | Train {tr_loss:.4f} | Val {va_loss:.4f} | "
              f"Time {fmt(ep_time)} | ETA {fmt(remaining)}")

        if va_loss < best:
            best = va_loss
            patience = 0
            torch.save({
                'model': model.state_dict(),
                'opt': optimizer.state_dict(),
                'scaler': scaler.state_dict(),
                'val_loss': best,
                'epoch': ep,
                'fold': fold_idx,
                'method': method_id,
                'stopped_reason': 'unknown'  # 学習中は不明
            }, save_path)
            print(f"  ✅ Save: {save_path} (best {best:.4f})")
            
            # 🆕 ベストモデル保存時に進捗も更新（クラッシュしても記録が残る）
            update_progress_best(progress, method_id, fold_idx, best, ep)
        else:
            patience += 1
            if patience >= EARLY_STOP_PATIENCE:
                print(f"⏹️ Early stop at epoch {ep}")
                stopped_reason = 'early_stop'
                break
        
        # 最終エポック到達
        if ep == NUM_EPOCHS:
            stopped_reason = 'completed'

    # 最終的な停止理由を記録
    if save_path.exists():
        checkpoint = torch.load(save_path, map_location='cpu')
        checkpoint['stopped_reason'] = stopped_reason
        checkpoint['final_epoch'] = ep  # 実際に到達したエポック
        torch.save(checkpoint, save_path)

    total_time = time.perf_counter() - start_all
    def fmt(t): 
        m, s = divmod(int(t), 60)
        h, m = divmod(m, 60)
        return f"{h:02d}:{m:02d}:{s:02d}"
    
    print(f"[M{method_id}-F{fold_idx}] 完了. Total {fmt(total_time)} | Best Val {best:.4f} | 終了理由: {stopped_reason}")
    return best, stopped_reason

print("✓ 学習ループ定義完了")


✓ 学習ループ定義完了


## 8. 進捗確認・管理（オプション）

### 8-A. 進捗確認・検証

進捗状態とモデルファイルの整合性を確認します。不整合があれば自動修復します。


In [7]:
# ===== 進捗確認・検証 =====
print("=" * 80)
print("📊 進捗状態の確認・検証")
print("=" * 80)

# 進捗ファイルの存在確認
PROGRESS_CHECK_FILE = Path("cache/cv_progress.json")
if not PROGRESS_CHECK_FILE.exists():
    print("\n✅ 進捗ファイルが存在しません（未実行または完全リセット済み）")
    print("   次回実行時は最初から開始されます。")
else:
    # 進捗を読み込み
    with open(PROGRESS_CHECK_FILE, 'r') as f:
        progress_check = json.load(f)
    
    print(f"\n📅 実行情報:")
    print(f"   開始時刻: {progress_check.get('started_at', 'N/A')}")
    print(f"   最終更新: {progress_check.get('last_update', 'N/A')}")
    print(f"\n💡 JSON記録タイミング:")
    print(f"   ✅ 学習開始時: 即座に記録（in_progress）")
    print(f"   ✅ ベストモデル更新時: エポック数と最良lossを更新")
    print(f"   ✅ 学習完了時: completedに移動")
    print(f"   → 途中でクラッシュしても、in_progressに記録が残ります")
    
    total_tasks = NUM_FOLDS * len([1, 2, 3])
    completed_tasks = len(progress_check.get('completed', {}))
    in_progress_tasks = len(progress_check.get('in_progress', {}))
    
    print(f"\n📈 進捗状況:")
    print(f"   総タスク数: {total_tasks}")
    print(f"   完了タスク: {completed_tasks}")
    print(f"   実行中: {in_progress_tasks}（途中で中断されたタスク）")
    print(f"   完了率: {completed_tasks / total_tasks * 100:.1f}%")
    
    # in_progressタスクの詳細表示
    if in_progress_tasks > 0:
        print(f"\n⚠️ 実行中のタスク（未完了）:")
        for task_key, task_info in progress_check.get('in_progress', {}).items():
            current_epoch = task_info.get('current_epoch', 0)
            current_loss = task_info.get('current_best_loss', float('inf'))
            started = task_info.get('started_at', 'N/A')
            last_update = task_info.get('last_update', 'N/A')
            print(f"   🔄 {task_key}:")
            print(f"      開始: {started}")
            print(f"      最終更新: {last_update}")
            print(f"      進捗: Epoch {current_epoch}/{NUM_EPOCHS} | Best Loss: {current_loss:.4f}")
            print(f"      → 次回実行時に続きから再開されます")
    
    # 各タスクの検証
    print(f"\n🔍 モデルファイル検証:")
    valid_count = 0
    invalid_count = 0
    
    for task_key, task_info in progress_check.get('completed', {}).items():
        # task_key例: "method1_fold0"
        parts = task_key.split('_')
        method_id = int(parts[0].replace('method', ''))
        fold_idx = int(parts[1].replace('fold', ''))
        
        # モデルファイル検証
        model_path = Path("model/cv_300ep") / f"method{method_id}_fold{fold_idx}_best.pth"
        
        if not model_path.exists():
            print(f"   ❌ {task_key}: モデルファイルが見つかりません")
            invalid_count += 1
        else:
            try:
                checkpoint = torch.load(model_path, map_location='cpu')
                if 'model' in checkpoint and 'epoch' in checkpoint:
                    file_size_mb = model_path.stat().st_size / (1024*1024)
                    stopped_reason = checkpoint.get('stopped_reason', 'unknown')
                    final_epoch = checkpoint.get('final_epoch', checkpoint.get('epoch', 0))
                    
                    # 終了理由をチェック
                    if stopped_reason == 'unknown' and final_epoch < NUM_EPOCHS:
                        print(f"   ❌ {task_key}: 途中で中断 (epoch={final_epoch}/{NUM_EPOCHS}, reason=unknown)")
                        invalid_count += 1
                    else:
                        status_emoji = "🏁" if stopped_reason == 'completed' else "⏹️" if stopped_reason == 'early_stop' else "✅"
                        print(f"   {status_emoji} {task_key}: OK (epoch={final_epoch}, loss={task_info['best_val_loss']:.4f}, {stopped_reason}, {file_size_mb:.1f}MB)")
                        valid_count += 1
                else:
                    print(f"   ❌ {task_key}: チェックポイントが不完全")
                    invalid_count += 1
            except Exception as e:
                print(f"   ❌ {task_key}: 読み込みエラー - {str(e)[:50]}")
                invalid_count += 1
    
    print(f"\n📊 検証結果:")
    print(f"   正常: {valid_count}")
    print(f"   異常: {invalid_count}")
    
    if invalid_count > 0:
        print(f"\n⚠️ {invalid_count}個のタスクに問題があります")
        print(f"   次回実行時に自動的に再学習されます")
    else:
        print(f"\n✅ 全てのタスクが正常です")

print("\n" + "=" * 80)


📊 進捗状態の確認・検証

📅 実行情報:
   開始時刻: 2025-11-16 23:27:35
   最終更新: 2026-03-23 22:06:55

💡 JSON記録タイミング:
   ✅ 学習開始時: 即座に記録（in_progress）
   ✅ ベストモデル更新時: エポック数と最良lossを更新
   ✅ 学習完了時: completedに移動
   → 途中でクラッシュしても、in_progressに記録が残ります

📈 進捗状況:
   総タスク数: 15
   完了タスク: 25
   実行中: 1（途中で中断されたタスク）
   完了率: 166.7%

⚠️ 実行中のタスク（未完了）:
   🔄 method6_fold0:
      開始: 2026-03-23 22:06:55
      最終更新: N/A
      進捗: Epoch 0/300 | Best Loss: inf
      → 次回実行時に続きから再開されます

🔍 モデルファイル検証:
   ⏹️ method1_fold0: OK (epoch=83, loss=0.0857, early_stop, 215.3MB)
   ⏹️ method2_fold0: OK (epoch=49, loss=0.0146, early_stop, 214.5MB)
   ⏹️ method3_fold0: OK (epoch=112, loss=0.1045, early_stop, 214.5MB)
   ⏹️ method1_fold1: OK (epoch=125, loss=0.0653, early_stop, 215.3MB)
   ⏹️ method2_fold1: OK (epoch=49, loss=0.0136, early_stop, 214.5MB)
   ⏹️ method3_fold1: OK (epoch=137, loss=0.1134, early_stop, 214.5MB)
   ⏹️ method1_fold2: OK (epoch=98, loss=0.0866, early_stop, 215.3MB)
   ⏹️ method2_fold2: OK (epoch=52, loss=0.0148, early_stop

### 8-B. 進捗リセット

**⚠️ 全体を最初からやり直したい場合のみ実行してください**

このセルを実行すると、保存された進捗情報がリセットされます。


In [8]:
# ===== 進捗リセット（オプション） =====
RESET_PROGRESS = False  # True にすると進捗をリセット

if RESET_PROGRESS:
    PROGRESS_FILE_RESET = Path("cache/cv_progress.json")
    
    if PROGRESS_FILE_RESET.exists():
        # バックアップ作成
        backup_path = PROGRESS_FILE_RESET.parent / f"cv_progress_backup_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        import shutil
        shutil.copy(PROGRESS_FILE_RESET, backup_path)
        
        # 進捗削除
        PROGRESS_FILE_RESET.unlink()
        
        print("=" * 80)
        print("🔄 進捗リセット完了")
        print("=" * 80)
        print(f"✓ 進捗ファイルを削除しました: {PROGRESS_FILE_RESET}")
        print(f"✓ バックアップを作成しました: {backup_path}")
        print(f"\n次回実行時は最初から開始されます。")
    else:
        print("⚠️ 進捗ファイルが見つかりません（既にリセット済みまたは未実行）")
else:
    print("💡 進捗リセットは無効です")
    print("   リセットする場合は RESET_PROGRESS = True に変更してください")


💡 進捗リセットは無効です
   リセットする場合は RESET_PROGRESS = True に変更してください


## 10. 5-Fold Cross-Validation 実行（Method1/2/3）

**Resume機能付き**: 途中で中断しても、再度実行すれば続きから自動的に再開されます。

### 📝 進捗記録のタイミング（リアルタイム）
1. **学習開始時**: `in_progress`に即座に記録
2. **ベストモデル更新時**: エポック数と最良lossを更新
3. **学習完了時**: `completed`に移動
4. **途中で中断**: `in_progress`に記録が残る → 次回再学習

### 自動検証機能（完全性チェック）
- ✅ モデルファイルの存在確認
- ✅ ファイル破損チェック
- ✅ **終了理由の検証**（early_stop / completed / unknown）
- ✅ **途中中断の自動検出**（unknown かつ NUM_EPOCHS 未到達）
- 🔄 不完全なタスクは自動的に再学習


In [9]:
# ===== Resume機能: 進捗管理 =====
PROGRESS_FILE = Path("cache/cv_progress.json")

def load_progress():
    """進捗状態を読み込み"""
    if PROGRESS_FILE.exists():
        with open(PROGRESS_FILE, 'r') as f:
            return json.load(f)
    return {
        "completed": {},      # 完了したタスク
        "in_progress": {},    # 実行中のタスク（途中で中断された）
        "started_at": None, 
        "last_update": None
    }

def save_progress(progress):
    """進捗状態を保存"""
    progress["last_update"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    PROGRESS_FILE.parent.mkdir(parents=True, exist_ok=True)
    with open(PROGRESS_FILE, 'w') as f:
        json.dump(progress, f, indent=2)

def verify_model_file(method_id, fold_idx):
    """モデルファイルが正常に存在するか検証"""
    model_path = MODEL_DIR / f"method{method_id}_fold{fold_idx}_best.pth"
    
    if not model_path.exists():
        return False, "ファイルが存在しません"
    
    try:
        # モデルファイルを読み込んでみる（破損チェック）
        checkpoint = torch.load(model_path, map_location='cpu')
        
        # 必要なキーが存在するか確認
        if 'model' not in checkpoint:
            return False, "モデルstate_dictが見つかりません"
        
        if 'epoch' not in checkpoint or 'val_loss' not in checkpoint:
            return False, "チェックポイント情報が不完全です"
        
        # モデルのサイズチェック（極端に小さい場合は破損の可能性）
        if model_path.stat().st_size < 1000000:  # 1MB未満は異常
            return False, f"ファイルサイズが異常に小さい ({model_path.stat().st_size} bytes)"
        
        # 終了理由をチェック（重要！）
        stopped_reason = checkpoint.get('stopped_reason', 'unknown')
        
        if stopped_reason == 'unknown':
            # 途中で異常終了した可能性
            final_epoch = checkpoint.get('final_epoch', checkpoint.get('epoch', 0))
            
            # 最終エポックまで到達していない、かつearly_stopでもない場合は異常
            if final_epoch < NUM_EPOCHS:
                return False, f"途中で中断された可能性 (epoch={final_epoch}, reason=unknown)"
        
        return True, checkpoint
    
    except Exception as e:
        return False, f"ファイル読み込みエラー: {str(e)}"

def is_completed(progress, method_id, fold_idx):
    """指定されたメソッド×Foldが完了済みか確認（モデルファイルも検証）"""
    key = f"method{method_id}_fold{fold_idx}"
    
    # in_progressに存在する場合は未完了（途中で中断された）
    if key in progress.get("in_progress", {}):
        print(f"    🔄 Method{method_id}-Fold{fold_idx}: 前回途中で中断されました")
        in_progress_info = progress["in_progress"][key]
        current_epoch = in_progress_info.get('current_epoch', 0)
        current_loss = in_progress_info.get('current_best_loss', float('inf'))
        print(f"       前回進捗: Epoch {current_epoch} | Best Loss: {current_loss:.4f}")
        print(f"       → 最初から再学習します")
        
        # in_progressから削除（再学習のため）
        del progress["in_progress"][key]
        save_progress(progress)
        return False
    
    # 進捗JSONのcompletedに記録されているか
    if key not in progress.get("completed", {}):
        return False
    
    # モデルファイルが正常に存在するか検証
    is_valid, result = verify_model_file(method_id, fold_idx)
    
    if not is_valid:
        # モデルファイルに問題がある場合は進捗から削除
        print(f"    ⚠️ Method{method_id}-Fold{fold_idx}: 進捗に記録されていますが、モデルファイルに問題があります")
        print(f"       理由: {result}")
        print(f"       → 進捗から削除して再学習します")
        
        del progress["completed"][key]
        save_progress(progress)
        return False
    
    return True

def mark_in_progress(progress, method_id, fold_idx):
    """学習開始を記録（in_progress状態）"""
    key = f"method{method_id}_fold{fold_idx}"
    
    # in_progressセクションがなければ作成
    if "in_progress" not in progress:
        progress["in_progress"] = {}
    
    progress["in_progress"][key] = {
        "started_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "current_epoch": 0,
        "current_best_loss": float('inf')
    }
    save_progress(progress)
    print(f"  📝 学習開始を記録: Method {method_id} - Fold {fold_idx}")

def update_progress_best(progress, method_id, fold_idx, best_val_loss, epoch):
    """ベストモデル更新時に進捗を更新"""
    key = f"method{method_id}_fold{fold_idx}"
    
    if "in_progress" not in progress:
        progress["in_progress"] = {}
    
    if key in progress["in_progress"]:
        progress["in_progress"][key]["current_epoch"] = int(epoch)
        progress["in_progress"][key]["current_best_loss"] = float(best_val_loss)
        progress["in_progress"][key]["last_update"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        save_progress(progress)

def mark_completed(progress, method_id, fold_idx, best_val_loss, epoch, stopped_reason='completed'):
    """メソッド×Foldを完了としてマーク（in_progressから削除してcompletedに移動）"""
    key = f"method{method_id}_fold{fold_idx}"
    
    # in_progressから削除
    if "in_progress" in progress and key in progress["in_progress"]:
        del progress["in_progress"][key]
    
    # completedに追加
    progress["completed"][key] = {
        "best_val_loss": float(best_val_loss),
        "epoch": int(epoch),
        "stopped_reason": stopped_reason,
        "completed_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
    save_progress(progress)

# 進捗状態をロード
progress = load_progress()

# 学習するメソッドの設定
TRAIN_METHODS = [1, 2, 3, 4, 5, 6]  # 1:Eyelid+Ellipse, 2:Edge, 3:6-class, 4:6-class(CE+Dice), 5:3-class(Amodal), 6:4-class(Visible-only)

# 結果を保存する辞書
fold_results = {1: [], 2: [], 3: [], 4: [], 5: [], 6: []}  # method_id -> list of fold results

if progress["started_at"] is None:
    progress["started_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    save_progress(progress)
    print("=" * 80)
    print("🆕 新規実行: 5-Fold Cross-Validation 開始")
    print("=" * 80)
else:
    print("=" * 80)
    print("🔄 Resume: 前回の続きから実行")
    print(f"   開始時刻: {progress['started_at']}")
    print(f"   前回更新: {progress['last_update']}")
    print(f"   完了済み: {len(progress['completed'])} / {NUM_FOLDS * len(TRAIN_METHODS)} タスク")
    print("=" * 80)
print(f"\n学習メソッド: {TRAIN_METHODS}")
print(f"総タスク数: {NUM_FOLDS * len(TRAIN_METHODS)} (Folds × Methods)")
print("=" * 80)

for fold_idx in range(NUM_FOLDS):
    print(f"\n{'='*80}")
    print(f"Fold {fold_idx} / {NUM_FOLDS-1}")
    print(f"{'='*80}")
    
    # データセット準備
    train_indices = fold_indices[str(fold_idx)]['train']
    val_indices   = fold_indices[str(fold_idx)]['val']
    
    train_paths = [image_paths[i] for i in train_indices]
    val_paths   = [image_paths[i] for i in val_indices]
    
    train_ds = EyeSegmentationDataset(train_paths, LABEL_SEG_DIR, LABEL_OBB_DIR, transform=True)
    val_ds   = EyeSegmentationDataset(val_paths,   LABEL_SEG_DIR, LABEL_OBB_DIR, transform=False)
    
    # 高速化: num_workers並列化 + pin_memory
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, 
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    
    print(f"  Train: {len(train_ds)} samples, Val: {len(val_ds)} samples")
    
    # 各メソッドについて学習
    for method_id in TRAIN_METHODS:
        # Resume: 完了済みチェック（モデルファイル検証含む）
        if is_completed(progress, method_id, fold_idx):
            completed_info = progress["completed"][f"method{method_id}_fold{fold_idx}"]
            stopped_reason = completed_info.get('stopped_reason', 'unknown')
            
            status_emoji = "🏁" if stopped_reason == 'completed' else "⏹️" if stopped_reason == 'early_stop' else "✅"
            print(f"\n  {status_emoji} Method {method_id} - Fold {fold_idx}: 完了済みスキップ")
            print(f"     Best Val Loss: {completed_info['best_val_loss']:.4f} (Epoch {completed_info['epoch']})")
            print(f"     終了理由: {stopped_reason}")
            print(f"     完了時刻: {completed_info['completed_at']}")
            
            # 結果に追加（集計用）
            fold_results[method_id].append({
                'method': method_id,
                'fold': fold_idx,
                'train_size': len(train_ds),
                'val_size': len(val_ds),
                'best_val_loss': completed_info['best_val_loss'],
                'stopped_reason': stopped_reason
            })
            continue
        
        print(f"\n  --- Method {method_id} 学習開始 ---")
        
        # モデル・損失関数・Optimizer初期化
        import time
        start_init = time.time()
        
        print(f"    モデル初期化中...", end=" ", flush=True)
        if method_id == 1:
            model = UNetMethod1().to(device)
            criterion = LossFunction1(lambda_ellipse=1.0)
        elif method_id == 2:
            model = UNetMethod2().to(device)
            criterion = LossFunction2(pos_weight=3.0)
        elif method_id == 3:
            model = UNetMethod3().to(device)
            criterion = LossFunction3()
        elif method_id == 4:
            model = UNetMethod4().to(device)
            criterion = LossFunction4()
        elif method_id == 5:
            model = UNetMethod5().to(device)
            criterion = LossFunction5()
        else:  # method_id == 6
            model = UNetMethod6().to(device)
            criterion = LossFunction6()
        print(f"完了 ({time.time() - start_init:.1f}秒)")
        
        print(f"    Optimizer初期化中...", end=" ", flush=True)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
        scaler = GradScaler()
        print(f"完了 ({time.time() - start_init:.1f}秒)")
        
        # DataLoaderの準備確認
        if NUM_WORKERS > 0:
            print(f"    💡 DataLoaderワーカー起動中（num_workers={NUM_WORKERS}）...")
            print(f"       Windowsでは30秒-2分かかる場合があります")
        
        # 学習実行（progressを渡して途中経過も記録）
        best_val_loss, stopped_reason = run_train_fold(model, train_loader, val_loader, criterion, optimizer, scaler, 
                                                        device, method_id, fold_idx, progress, show_progress=True)
        
        # 結果保存
        fold_results[method_id].append({
            'method': method_id,
            'fold': fold_idx,
            'train_size': len(train_ds),
            'val_size': len(val_ds),
            'best_val_loss': best_val_loss,
            'stopped_reason': stopped_reason
        })
        
        # Resume: 進捗を記録（正常終了の場合のみ）
        if stopped_reason in ['early_stop', 'completed']:
            model_path = MODEL_DIR / f"method{method_id}_fold{fold_idx}_best.pth"
            if model_path.exists():
                checkpoint = torch.load(model_path, map_location='cpu')
                final_epoch = checkpoint.get('final_epoch', checkpoint.get('epoch', NUM_EPOCHS))
                mark_completed(progress, method_id, fold_idx, best_val_loss, final_epoch, stopped_reason)
                print(f"  📝 進捗保存: Method {method_id} - Fold {fold_idx} 完了 ({stopped_reason})")
        else:
            print(f"  ⚠️ 異常終了を検出: Method {method_id} - Fold {fold_idx}")
            print(f"     進捗には記録しません（次回再学習されます）")
        
        # メモリクリア
        del model, optimizer, scaler, criterion
        torch.cuda.empty_cache()
        
        print(f"  --- Method {method_id} 完了 | Best Val Loss: {best_val_loss:.4f} ---\n")
    
    # Fold終了後のクリア
    del train_loader, val_loader, train_ds, val_ds
    torch.cuda.empty_cache()
    
    print(f"\n{'='*80}")
    print(f"Fold {fold_idx} 完了")
    print(f"{'='*80}\n")

print("\n" + "=" * 80)
print("✅ 全Fold学習完了")
print("=" * 80)

# 完了統計
total_tasks = NUM_FOLDS * len(TRAIN_METHODS)
completed_tasks = len(progress["completed"])
in_progress_count = len(progress.get("in_progress", {}))

print(f"\n📊 完了統計:")
print(f"   総タスク数: {total_tasks}")
print(f"   完了タスク: {completed_tasks}")
print(f"   実行中: {in_progress_count}（途中で中断）")
print(f"   完了率: {completed_tasks / total_tasks * 100:.1f}%")

if completed_tasks == total_tasks:
    print(f"\n🎉 全タスク完了！")
    print(f"   開始時刻: {progress['started_at']}")
    print(f"   終了時刻: {progress['last_update']}")
    
    # 次回実行のために進捗ファイルをリセットするかの注意書き
    print(f"\n💡 ヒント:")
    print(f"   再度全体を実行する場合は、以下のファイルを削除してください:")
    print(f"   - {PROGRESS_FILE}")
    print(f"   - {MODEL_DIR}/*.pth (オプション)")


🔄 Resume: 前回の続きから実行
   開始時刻: 2025-11-16 23:27:35
   前回更新: 2026-03-23 22:06:55
   完了済み: 25 / 30 タスク

学習メソッド: [1, 2, 3, 4, 5, 6]
総タスク数: 30 (Folds × Methods)

Fold 0 / 4
  📦 楕円キャッシュをロード中: cache\ellipse_params\ellipse_params.npz
  ✓ 3950 個の楕円パラメータをロードしました
  ✓ Method1の学習が約25%高速化されます
  Train: 1593 samples, Val: 399 samples

  ⏹️ Method 1 - Fold 0: 完了済みスキップ
     Best Val Loss: 0.0857 (Epoch 83)
     終了理由: early_stop
     完了時刻: 2025-11-19 02:29:59

  ⏹️ Method 2 - Fold 0: 完了済みスキップ
     Best Val Loss: 0.0146 (Epoch 49)
     終了理由: early_stop
     完了時刻: 2025-11-19 05:44:13

  ⏹️ Method 3 - Fold 0: 完了済みスキップ
     Best Val Loss: 0.1045 (Epoch 112)
     終了理由: early_stop
     完了時刻: 2025-11-19 11:18:17

  ⏹️ Method 4 - Fold 0: 完了済みスキップ
     Best Val Loss: 0.0848 (Epoch 79)
     終了理由: early_stop
     完了時刻: 2026-02-21 05:07:33

  ⏹️ Method 5 - Fold 0: 完了済みスキップ
     Best Val Loss: 0.0528 (Epoch 70)
     終了理由: early_stop
     完了時刻: 2026-02-25 11:05:19
    🔄 Method6-Fold0: 前回途中で中断されました
       前回進捗: Epoch 0 

C:\Users\CorneAI\AppData\Local\Temp\ipykernel_125744\1242403490.py:246: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Train:   0%|          | 0/100 [00:00<?, ?it/s]

C:\Users\CorneAI\AppData\Local\Temp\ipykernel_125744\1852268689.py:11: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Valid:   0%|          | 0/25 [00:00<?, ?it/s]

...

Valid:   0%|          | 0/25 [00:00<?, ?it/s]

[M6-F4] Epoch 050/300 | Train 0.0182 | Val 0.0387 | Time 00:03:28 | ETA 14:35:19


Train:   0%|          | 0/100 [00:00<?, ?it/s]

Valid:   0%|          | 0/25 [00:00<?, ?it/s]

[M6-F4] Epoch 051/300 | Train 0.0167 | Val 0.0425 | Time 00:03:28 | ETA 14:31:41
⏹️ Early stop at epoch 51
[M6-F4] 完了. Total 02:58:35 | Best Val 0.0366 | 終了理由: early_stop
  📝 進捗保存: Method 6 - Fold 4 完了 (early_stop)
  --- Method 6 完了 | Best Val Loss: 0.0366 ---


Fold 4 完了


✅ 全Fold学習完了

📊 完了統計:
   総タスク数: 30
   完了タスク: 30
   実行中: 0（途中で中断）
   完了率: 100.0%

🎉 全タスク完了！
   開始時刻: 2025-11-16 23:27:35
   終了時刻: 2026-03-24 14:12:14

💡 ヒント:
   再度全体を実行する場合は、以下のファイルを削除してください:
   - cache\cv_progress.json
   - model\cv_300ep/*.pth (オプション)


## 11. 評価（各Foldごと）

学習済みモデルを読み込んで、各Foldの評価を実行します。

### 計算される指標（MetricsReloaded 公式実装）

各画像について、eyelid / iris / pupil それぞれに対して：

- **DSC** (Dice Similarity Coefficient) — 既存
- **HD95** (Hausdorff Distance 95%ile) — 境界の外れ値耐性指標
- **NSD** (Normalized Surface Distance) — τ 以内に境界がある割合

### 出力 CSV

| Method | ファイル | モード列の値 |
|---|---|---|
| 1 | `results/cv_method1_reloaded_perimage_{ts}.csv` | `ellipse_regression` |
| 2 | `results/cv_method2_reloaded_perimage_{ts}.csv` | `edge_ellipse_fit` |
| 3 | `results/cv_method3_full_vs_exposed_perimage_{ts}.csv` | `raw` / `outerarc` / `fullmax` / `ransac_whole` |
| 4 | `results/cv_method4_full_vs_exposed_perimage_{ts}.csv` | 同上 |
| 5 | `results/cv_method5_amodal_perimage_{ts}.csv` | `raw` / `fullmax` / `ransac_whole` |
| 6 | `results/cv_method6_visible_boundary_perimage_{ts}.csv` | `raw` / `boundary` / `fullmax` |

### スキーマ

```
filename, subject_id, mode, fold,
eyelid, iris, pupil, mean,                    # DSC（Dice）
eyelid_hd95, iris_hd95, pupil_hd95,           # HD95 (px)
eyelid_nsd, iris_nsd, pupil_nsd               # NSD (τ=2/2/1 px)
```

### 引用
> Maier-Hein L, Reinke A, et al. Metrics reloaded: recommendations for image analysis validation. *Nat Methods* 2024;21:195–212.


In [ ]:
# ===== 評価用ユーティリティ =====
from skimage.measure import EllipseModel, ransac

# === MetricsReloaded patch ===
import sys, os
_repo_root = os.path.abspath('.')
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)
from evaluation.metrics_reloaded_eval import compute_hd95_nsd, DEFAULT_NSD_TAU

def _augment_hd95_nsd(row, lid_pred, iris_pred, pupil_pred, gt_lid, gt_iris, gt_pupil):
    """Add hd95 / nsd columns for each structure to a per-image row dict."""
    h, n = compute_hd95_nsd(lid_pred, gt_lid, DEFAULT_NSD_TAU['eyelid'])
    row['eyelid_hd95'] = h; row['eyelid_nsd'] = n
    h, n = compute_hd95_nsd(iris_pred, gt_iris, DEFAULT_NSD_TAU['iris'])
    row['iris_hd95'] = h; row['iris_nsd'] = n
    h, n = compute_hd95_nsd(pupil_pred, gt_pupil, DEFAULT_NSD_TAU['pupil'])
    row['pupil_hd95'] = h; row['pupil_nsd'] = n
# === end MetricsReloaded patch ===


def dice_binary_np(pred_bin_255: np.ndarray, gt_bin_255: np.ndarray, smooth=1e-6):
    """Numpy配列でDice係数を計算"""
    p = (pred_bin_255>0).astype(np.uint8)
    g = (gt_bin_255>0).astype(np.uint8)
    inter = (p & g).sum()
    union = p.sum() + g.sum()
    return (2*inter + smooth) / (union + smooth)

def mask_to_edge(mask_bin: np.ndarray, thickness: int = 3) -> np.ndarray:
    """0/255マスク → 輪郭(0/255) with specified thickness"""
    contours, _ = cv2.findContours((mask_bin>0).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    edge = np.zeros_like(mask_bin, dtype=np.uint8)
    for cnt in contours:
        cv2.drawContours(edge, [cnt], -1, 255, thickness=thickness)
    return edge

def ellipse_params_to_mask(params: np.ndarray, H: int, W: int) -> np.ndarray:
    """楕円パラメータ(5,) -> マスク(H,W) 0/255"""
    cx = params[0]*W
    cy = params[1]*H
    w  = params[2]*W
    h  = params[3]*H
    angle = params[4]*180.0
    mask = np.zeros((H,W), dtype=np.uint8)
    center = (int(cx), int(cy))
    axes   = (max(1,int(w/2)), max(1,int(h/2)))
    cv2.ellipse(mask, center, axes, angle, 0, 360, 255, thickness=-1)
    return mask

def bin_edge_to_filled(edge_bin: np.ndarray) -> np.ndarray:
    """エッジ -> 塗りつぶしマスク"""
    edge_uint8 = (edge_bin > 0).astype(np.uint8)
    kernel = np.ones((25, 25), np.uint8)
    closed = cv2.morphologyEx(edge_uint8, cv2.MORPH_CLOSE, kernel, iterations=6)
    contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    filled = np.zeros_like(edge_bin, dtype=np.uint8)
    if len(contours) > 0:
        largest_contour = max(contours, key=cv2.contourArea)
        cv2.drawContours(filled, [largest_contour], -1, 255, thickness=-1)
    return filled

def fit_ellipse_ransac(edge_points, min_samples=5, residual_threshold=2.0, max_trials=100):
    """
    RANSAC + 最小二乗法による楕円フィッティング
    部分的なエッジ（90度未満）に対してロバスト

    Args:
        edge_points: (N, 2) array of (y, x) coordinates
        min_samples: RANSACの最小サンプル数（楕円は5点必要）
        residual_threshold: インライア判定の閾値（ピクセル単位）
        max_trials: RANSAC試行回数

    Returns:
        cv2.ellipse形式のタプル: ((cx, cy), (width, height), angle_deg)
        失敗時: None
    """
    if len(edge_points) < 5:
        return None

    # (y, x) -> (x, y) に変換（skimageは(x,y)形式）
    points = edge_points[:, ::-1].astype(np.float64)

    try:
        # RANSACで楕円をフィット
        model, inliers = ransac(
            points,
            EllipseModel,
            min_samples=min_samples,
            residual_threshold=residual_threshold,
            max_trials=max_trials
        )

        # インライア点のみで再フィッティング（より精密）
        if inliers is not None and np.sum(inliers) >= 5:
            final_model = EllipseModel()
            final_model.estimate(points[inliers])

            # パラメータ抽出: (xc, yc, a, b, theta)
            xc, yc, a, b, theta = final_model.params

            # cv2.ellipse形式に変換: ((xc, yc), (2*a, 2*b), theta_deg)
            return ((float(xc), float(yc)), (float(2*a), float(2*b)), float(np.degrees(theta)))

    except Exception as e:
        # RANSACが失敗した場合（データが少なすぎる等）
        pass

    return None

def binary_to_ellipse_params(mask_bin: np.ndarray, residual_threshold=2.0):
    """
    マスク(0/255)からRANSAC+最小二乗法で楕円フィッティング → (cx,cy,a,b,theta)を0-1正規化で返す。
    部分的なエッジに対してもロバストな推定を行う。
    失敗時はNone

    Args:
        mask_bin: 0/255のマスク
        residual_threshold: RANSACのインライア閾値（ピクセル単位、デフォルト=2.0）
    """
    ys, xs = np.where(mask_bin > 0)
    if len(xs) < 5:
        return None

    # RANSAC + 最小二乗法による楕円フィッティング
    # (y, x) formatでfit_ellipse_ransacに渡す
    edge_points = np.column_stack([ys, xs])
    ellipse = fit_ellipse_ransac(
        edge_points,
        residual_threshold=residual_threshold,
        max_trials=100
    )

    if ellipse is None:
        return None

    # cv2.ellipse形式: ((cx,cy),(w,h),angle_deg)
    (cx,cy), (w,h), angle = ellipse

    # 正規化 (a,bは直径→0..1)
    cx_n = np.clip(cx / mask_bin.shape[1], 0, 1)
    cy_n = np.clip(cy / mask_bin.shape[0], 0, 1)
    a_n  = np.clip(w  / mask_bin.shape[1], 1e-6, 1.0)
    b_n  = np.clip(h  / mask_bin.shape[0], 1e-6, 1.0)
    # 角度を[0,180)→[0,1]
    theta_n = (angle % 180) / 180.0
    return np.array([cx_n, cy_n, a_n, b_n, theta_n], dtype=np.float32)

# ===== Method別評価関数 =====
@torch.no_grad()
def evaluate_method1(model, val_loader, device):
    """Method1評価: Eyelid, Iris, Pupilの3つのDice + HD95 + NSD (per-image)"""
    model.eval()
    lid_scores, iris_scores, pupil_scores = [], [], []
    per_rows = []

    for batch in tqdm(val_loader, desc="M1評価", leave=False):
        img = batch['image'].to(device)
        gt_lid   = batch['mask_lid'].cpu().numpy()
        gt_iris  = batch['mask_iris'].cpu().numpy()
        gt_pupil = batch['mask_pupil'].cpu().numpy()
        filenames = batch.get('filename', None)

        with autocast():
            out = model(img)

        for b in range(img.shape[0]):
            # Eyelid
            lid_logits = out['eyelid_seg'][b:b+1]
            lid_pred = (torch.sigmoid(lid_logits).cpu().squeeze().numpy() >= 0.5).astype(np.uint8)*255
            lid_d = float(dice_binary_np(lid_pred, gt_lid[b]))
            lid_scores.append(lid_d)

            # Iris/Pupil ellipse
            iris_params  = torch.sigmoid(out['iris_ellipse']).cpu().numpy()[b]
            pupil_params = torch.sigmoid(out['pupil_ellipse']).cpu().numpy()[b]
            iris_mask  = ellipse_params_to_mask(iris_params,  IMAGE_HEIGHT, IMAGE_WIDTH)
            pupil_mask = ellipse_params_to_mask(pupil_params, IMAGE_HEIGHT, IMAGE_WIDTH)
            iris_d = float(dice_binary_np(iris_mask, gt_iris[b]))
            pupil_d = float(dice_binary_np(pupil_mask, gt_pupil[b]))
            iris_scores.append(iris_d); pupil_scores.append(pupil_d)

            filename = filenames[b] if filenames is not None else str(b)
            subject_id = str(filename).split('-', 1)[0]
            row = {
                'filename': str(filename), 'subject_id': str(subject_id),
                'mode': 'ellipse_regression',
                'eyelid': lid_d, 'iris': iris_d, 'pupil': pupil_d,
                'mean': float(np.mean([lid_d, iris_d, pupil_d])),
            }
            _augment_hd95_nsd(row, lid_pred, iris_mask, pupil_mask,
                              gt_lid[b], gt_iris[b], gt_pupil[b])
            per_rows.append(row)

    result = {
        'lid': float(np.mean(lid_scores)),
        'iris': float(np.mean(iris_scores)),
        'pupil': float(np.mean(pupil_scores)),
        'mean': float(np.mean([np.mean(lid_scores), np.mean(iris_scores), np.mean(pupil_scores)])),
    }
    return result, per_rows

@torch.no_grad()
def evaluate_method2(model, val_loader, device):
    """Method2評価: Eyelid, Iris, Pupilの3つのDice + HD95 + NSD (per-image)"""
    model.eval()
    lid_scores, iris_scores, pupil_scores = [], [], []
    iris_scores_before_ellipse, pupil_scores_before_ellipse = [], []
    per_rows = []

    for batch in tqdm(val_loader, desc="M2評価", leave=False):
        img = batch['image'].to(device)
        gt_lid   = batch['mask_lid'].cpu().numpy()
        gt_iris  = batch['mask_iris'].cpu().numpy()
        gt_pupil = batch['mask_pupil'].cpu().numpy()
        filenames = batch.get('filename', None)

        with autocast():
            out = model(img)
            edge_logits = out['edge_logits']

        for b in range(img.shape[0]):
            # Eyelid: edge -> fill
            lid_edge = (torch.sigmoid(edge_logits[b,0:1]).cpu().squeeze().numpy() >= 0.5).astype(np.uint8)*255
            lid_fill = bin_edge_to_filled(lid_edge)
            lid_d = float(dice_binary_np(lid_fill, gt_lid[b]))
            lid_scores.append(lid_d)

            # Iris/Pupil: edge -> ellipse fit
            iris_edge  = (torch.sigmoid(edge_logits[b,1:2]).cpu().squeeze().numpy() >= 0.5).astype(np.uint8)*255
            pupil_edge = (torch.sigmoid(edge_logits[b,2:3]).cpu().squeeze().numpy() >= 0.5).astype(np.uint8)*255

            iris_fill_before = bin_edge_to_filled(iris_edge)
            pupil_fill_before = bin_edge_to_filled(pupil_edge)
            iris_scores_before_ellipse.append(float(dice_binary_np(iris_fill_before, gt_iris[b])))
            pupil_scores_before_ellipse.append(float(dice_binary_np(pupil_fill_before, gt_pupil[b])))

            iris_mask = np.zeros((IMAGE_HEIGHT, IMAGE_WIDTH), np.uint8)
            pupil_mask = np.zeros((IMAGE_HEIGHT, IMAGE_WIDTH), np.uint8)

            iris_pts = np.column_stack(np.where(iris_edge > 0))
            if len(iris_pts) >= 5:
                try:
                    ellipse = cv2.fitEllipse(iris_pts[:, ::-1].astype(np.int32))
                    cv2.ellipse(iris_mask, ellipse, 255, thickness=-1)
                except: pass

            pupil_pts = np.column_stack(np.where(pupil_edge > 0))
            if len(pupil_pts) >= 5:
                try:
                    ellipse = cv2.fitEllipse(pupil_pts[:, ::-1].astype(np.int32))
                    cv2.ellipse(pupil_mask, ellipse, 255, thickness=-1)
                except: pass

            iris_d = float(dice_binary_np(iris_mask, gt_iris[b]))
            pupil_d = float(dice_binary_np(pupil_mask, gt_pupil[b]))
            iris_scores.append(iris_d); pupil_scores.append(pupil_d)

            filename = filenames[b] if filenames is not None else str(b)
            subject_id = str(filename).split('-', 1)[0]
            row = {
                'filename': str(filename), 'subject_id': str(subject_id),
                'mode': 'edge_ellipse_fit',
                'eyelid': lid_d, 'iris': iris_d, 'pupil': pupil_d,
                'mean': float(np.mean([lid_d, iris_d, pupil_d])),
            }
            _augment_hd95_nsd(row, lid_fill, iris_mask, pupil_mask,
                              gt_lid[b], gt_iris[b], gt_pupil[b])
            per_rows.append(row)

    result = {
        'lid': float(np.mean(lid_scores)),
        'iris': float(np.mean(iris_scores)),
        'pupil': float(np.mean(pupil_scores)),
        'mean': float(np.mean([np.mean(lid_scores), np.mean(iris_scores), np.mean(pupil_scores)])),
        'iris_before_ellipse': float(np.mean(iris_scores_before_ellipse)),
        'pupil_before_ellipse': float(np.mean(pupil_scores_before_ellipse)),
    }
    return result, per_rows

def _max_contour_points_yx(mask_bin_255: np.ndarray):
    """0/255マスクから最大外輪郭の点群(y,x)を返す。なければ None"""
    m = (mask_bin_255 > 0).astype(np.uint8)
    contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours:
        return None
    cnt = max(contours, key=cv2.contourArea)
    pts = cnt.reshape(-1, 2)  # (N,2) in (x,y)
    return pts[:, ::-1].astype(np.int32)  # (y,x)


def ellipse_mask_from_fullmax_contour(vis_255: np.ndarray, occ_255: np.ndarray):
    """(vis,occ)をORして得たfullマスクから最大外輪郭のみを使ってcv2.fitEllipseで楕円化（RANSACなし）。"""
    full = ((vis_255 > 0) | (occ_255 > 0)).astype(np.uint8) * 255
    pts_yx = _max_contour_points_yx(full)
    if pts_yx is None or len(pts_yx) < 5:
        return np.zeros_like(full), None

    pts_xy = pts_yx[:, ::-1].astype(np.float32)
    try:
        ellipse = cv2.fitEllipse(pts_xy)
    except Exception:
        return np.zeros_like(full), None

    (cx, cy), (w, h), angle = ellipse
    H, W = full.shape
    mask = np.zeros((H, W), dtype=np.uint8)
    center = (int(round(cx)), int(round(cy)))
    axes = (max(1, int(round(w / 2))), max(1, int(round(h / 2))))
    cv2.ellipse(mask, center, axes, float(angle), 0, 360, 255, thickness=-1)
    return mask, ellipse


def ellipse_mask_from_visible_outer_arc(vis_255: np.ndarray, occ_255: np.ndarray,
                                       boundary_width: int = 7,
                                       vis_near_radius: int = 2,
                                       outer_thickness: int = 2):
    """OuterArc（露出弧fit, RANSACなし）

    full(vis|occ)の最大外輪郭を取り、vis/occ境界（=まぶた境界になりやすい）を除外しつつ
    vis近傍の外輪郭点だけでcv2.fitEllipseします。
    """
    H, W = vis_255.shape
    full = ((vis_255 > 0) | (occ_255 > 0)).astype(np.uint8)

    # full最大外輪郭エッジ
    contours, _ = cv2.findContours(full, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours:
        return np.zeros((H, W), np.uint8), None
    cnt = max(contours, key=cv2.contourArea)
    edge_outer = np.zeros((H, W), np.uint8)
    cv2.drawContours(edge_outer, [cnt], -1, 255, thickness=outer_thickness)

    # vis/occ境界付近（除外領域）
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    vis = (vis_255 > 0).astype(np.uint8)
    occ = (occ_255 > 0).astype(np.uint8)
    vis_d = cv2.dilate(vis, k, iterations=1)
    occ_d = cv2.dilate(occ, k, iterations=1)
    boundary = ((vis_d & occ_d) > 0).astype(np.uint8)
    boundary_d = cv2.dilate(boundary, k, iterations=max(1, boundary_width))

    allowed_outer = np.where(boundary_d > 0, 0, edge_outer)

    # vis近傍の外輪郭だけを採用
    if vis_near_radius > 0:
        vis_near = cv2.dilate(vis, k, iterations=vis_near_radius)
    else:
        vis_near = vis
    edge_arc = np.where(vis_near > 0, allowed_outer, 0)

    pts_yx = np.column_stack(np.where(edge_arc > 0))
    if len(pts_yx) < 5:
        return np.zeros((H, W), np.uint8), None

    pts_xy = pts_yx[:, ::-1].astype(np.float32)
    try:
        ellipse = cv2.fitEllipse(pts_xy)
    except Exception:
        return np.zeros((H, W), np.uint8), None

    (cx, cy), (w, h), angle = ellipse
    mask = np.zeros((H, W), dtype=np.uint8)
    center = (int(round(cx)), int(round(cy)))
    axes = (max(1, int(round(w / 2))), max(1, int(round(h / 2))))
    cv2.ellipse(mask, center, axes, float(angle), 0, 360, 255, thickness=-1)
    return mask, ellipse


@torch.no_grad()
def evaluate_method3(model, val_loader, device):
    """Method3評価: U-Net 6cls出力に対して

    - raw（6cls合成）
    - ransac_whole（既存：fullのエッジ→RANSAC楕円）
    - outerarc（露出弧fit, RANSACなし）
    - fullmax（full最大外輪郭fit, RANSACなし）

    を画像ごとに計算し、fold平均とper-image行を返す。

    Returns:
        result_dict, per_image_rows
    """
    model.eval()
    lid_scores, iris_scores, pupil_scores = [], [], []  # 既存（RANSAC楕円後）
    iris_scores_before_ellipse, pupil_scores_before_ellipse = [], []

    per_rows = []

    for batch in tqdm(val_loader, desc="M3評価", leave=False):
        img = batch['image'].to(device)
        gt_lid   = batch['mask_lid'].cpu().numpy()
        gt_iris  = batch['mask_iris'].cpu().numpy()
        gt_pupil = batch['mask_pupil'].cpu().numpy()
        filenames = batch.get('filename', None)

        with autocast():
            out = model(img)
            logits = out['five_class_seg']

        pred_labels = torch.argmax(logits, dim=1).cpu().numpy()

        for b in range(pred_labels.shape[0]):
            pred = pred_labels[b]
            filename = filenames[b] if filenames is not None else str(b)
            subject_id = str(filename).split('-', 1)[0]

            # Eyelid: 6クラスから合成 (class 1 ∪ 2 ∪ 4)
            lid_m3_bin = (((pred==1)|(pred==2)|(pred==4)).astype(np.uint8)*255)
            lid_d = float(dice_binary_np(lid_m3_bin, gt_lid[b]))
            lid_scores.append(lid_d)

            # vis/occ
            iris_vis = ((pred==2).astype(np.uint8)*255)
            iris_occ = ((pred==3).astype(np.uint8)*255)
            pupil_vis = ((pred==4).astype(np.uint8)*255)
            pupil_occ = ((pred==5).astype(np.uint8)*255)

            # raw（合成）
            iris_raw = (((pred==2)|(pred==3)).astype(np.uint8)*255)
            pupil_raw = (((pred==4)|(pred==5)).astype(np.uint8)*255)
            iris_raw_d = float(dice_binary_np(iris_raw, gt_iris[b]))
            pupil_raw_d = float(dice_binary_np(pupil_raw, gt_pupil[b]))
            mean_raw = float(np.mean([lid_d, iris_raw_d, pupil_raw_d]))

            iris_scores_before_ellipse.append(iris_raw_d)
            pupil_scores_before_ellipse.append(pupil_raw_d)

            # ransac_whole（既存実装と同じ）
            iris_edge = mask_to_edge(iris_raw, thickness=3)
            pupil_edge = mask_to_edge(pupil_raw, thickness=3)
            iris_params3 = binary_to_ellipse_params(iris_edge)
            pupil_params3 = binary_to_ellipse_params(pupil_edge)
            iris_ransac = ellipse_params_to_mask(iris_params3, IMAGE_HEIGHT, IMAGE_WIDTH) if iris_params3 is not None else np.zeros((IMAGE_HEIGHT,IMAGE_WIDTH), np.uint8)
            pupil_ransac = ellipse_params_to_mask(pupil_params3, IMAGE_HEIGHT, IMAGE_WIDTH) if pupil_params3 is not None else np.zeros((IMAGE_HEIGHT,IMAGE_WIDTH), np.uint8)
            iris_ransac_d = float(dice_binary_np(iris_ransac, gt_iris[b]))
            pupil_ransac_d = float(dice_binary_np(pupil_ransac, gt_pupil[b]))
            mean_ransac = float(np.mean([lid_d, iris_ransac_d, pupil_ransac_d]))

            iris_scores.append(iris_ransac_d)
            pupil_scores.append(pupil_ransac_d)

            # outerarc（露出弧fit）
            iris_outer, _ = ellipse_mask_from_visible_outer_arc(iris_vis, iris_occ, boundary_width=7, vis_near_radius=2, outer_thickness=2)
            pupil_outer, _ = ellipse_mask_from_visible_outer_arc(pupil_vis, pupil_occ, boundary_width=3, vis_near_radius=1, outer_thickness=2)
            iris_outer_d = float(dice_binary_np(iris_outer, gt_iris[b]))
            pupil_outer_d = float(dice_binary_np(pupil_outer, gt_pupil[b]))
            mean_outer = float(np.mean([lid_d, iris_outer_d, pupil_outer_d]))

            # fullmax（最大外輪郭fit）
            iris_fullmax, _ = ellipse_mask_from_fullmax_contour(iris_vis, iris_occ)
            pupil_fullmax, _ = ellipse_mask_from_fullmax_contour(pupil_vis, pupil_occ)
            iris_fullmax_d = float(dice_binary_np(iris_fullmax, gt_iris[b]))
            pupil_fullmax_d = float(dice_binary_np(pupil_fullmax, gt_pupil[b]))
            mean_fullmax = float(np.mean([lid_d, iris_fullmax_d, pupil_fullmax_d]))

            # per-image rows (long format) with HD95 + NSD
            for mode, i_d, p_d, m_d, i_mask, p_mask in [
                ('raw', iris_raw_d, pupil_raw_d, mean_raw, iris_raw, pupil_raw),
                ('outerarc', iris_outer_d, pupil_outer_d, mean_outer, iris_outer, pupil_outer),
                ('fullmax', iris_fullmax_d, pupil_fullmax_d, mean_fullmax, iris_fullmax, pupil_fullmax),
                ('ransac_whole', iris_ransac_d, pupil_ransac_d, mean_ransac, iris_ransac, pupil_ransac),
            ]:
                row = {
                    'filename': str(filename), 'subject_id': str(subject_id),
                    'mode': mode,
                    'eyelid': lid_d, 'iris': i_d, 'pupil': p_d, 'mean': m_d,
                }
                _augment_hd95_nsd(row, lid_m3_bin, i_mask, p_mask,
                                  gt_lid[b], gt_iris[b], gt_pupil[b])
                per_rows.append(row)

    result = {
        'lid': float(np.mean(lid_scores)) if len(lid_scores) else 0.0,
        'iris': float(np.mean(iris_scores)) if len(iris_scores) else 0.0,
        'pupil': float(np.mean(pupil_scores)) if len(pupil_scores) else 0.0,
        'mean': float(np.mean([np.mean(lid_scores), np.mean(iris_scores), np.mean(pupil_scores)])) if len(lid_scores) else 0.0,
        'iris_before_ellipse': float(np.mean(iris_scores_before_ellipse)) if len(iris_scores_before_ellipse) else 0.0,
        'pupil_before_ellipse': float(np.mean(pupil_scores_before_ellipse)) if len(pupil_scores_before_ellipse) else 0.0,
    }

    return result, per_rows


def _fit_ellipse_from_binary(mask_bin_255):
    """Binary mask (0/255) -> cv2 ellipse fit from largest contour.
    Returns: ((cx,cy),(w,h),angle_deg) or None if failed."""
    m = (mask_bin_255 > 0).astype(np.uint8)
    contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours:
        return None
    cnt = max(contours, key=cv2.contourArea)
    if len(cnt) < 5:
        return None
    try:
        return cv2.fitEllipse(cnt)
    except Exception:
        return None

@torch.no_grad()
def evaluate_method5(model, val_loader, device):
    """Method5評価: 3-class amodal sigmoid → eyelid/iris/pupil Dice

    Post-processing variants:
    - raw: threshold sigmoid > 0.5
    - fullmax: largest contour → fitEllipse (for iris/pupil)
    - ransac_whole: edge → RANSAC ellipse (for iris/pupil)

    Returns:
        result_dict (primary=fullmax), per_image_rows
    """
    model.eval()
    per_rows = []
    # Collect scores for primary metric (fullmax)
    lid_scores, iris_scores, pupil_scores = [], [], []

    for batch in tqdm(val_loader, desc="M5評価", leave=False):
        img = batch['image'].to(device)
        gt_lid   = batch['mask_lid'].cpu().numpy()
        gt_iris  = batch['mask_iris'].cpu().numpy()
        gt_pupil = batch['mask_pupil'].cpu().numpy()
        filenames = batch.get('filename', None)

        with autocast():
            out = model(img)
            logits = out['amodal_logits']  # (B,3,H,W)

        probs = torch.sigmoid(logits).cpu().numpy()  # (B,3,H,W)

        for b in range(probs.shape[0]):
            filename = filenames[b] if filenames is not None else str(b)
            subject_id = str(filename).split('-', 1)[0]

            # Raw predictions (threshold 0.5)
            lid_raw   = (probs[b, 0] >= 0.5).astype(np.uint8) * 255
            iris_raw  = (probs[b, 1] >= 0.5).astype(np.uint8) * 255
            pupil_raw = (probs[b, 2] >= 0.5).astype(np.uint8) * 255

            lid_d = float(dice_binary_np(lid_raw, gt_lid[b]))
            iris_raw_d = float(dice_binary_np(iris_raw, gt_iris[b]))
            pupil_raw_d = float(dice_binary_np(pupil_raw, gt_pupil[b]))
            mean_raw = float(np.mean([lid_d, iris_raw_d, pupil_raw_d]))

            # fullmax: largest contour → fitEllipse (iris/pupil only)
            iris_fullmax_mask = np.zeros_like(iris_raw)
            ell = _fit_ellipse_from_binary(iris_raw)
            if ell is not None:
                (cx, cy), (w, h), angle = ell
                center = (int(round(cx)), int(round(cy)))
                axes = (max(1, int(round(w / 2))), max(1, int(round(h / 2))))
                cv2.ellipse(iris_fullmax_mask, center, axes, float(angle), 0, 360, 255, thickness=-1)
            iris_fullmax_d = float(dice_binary_np(iris_fullmax_mask, gt_iris[b]))

            pupil_fullmax_mask = np.zeros_like(pupil_raw)
            ell = _fit_ellipse_from_binary(pupil_raw)
            if ell is not None:
                (cx, cy), (w, h), angle = ell
                center = (int(round(cx)), int(round(cy)))
                axes = (max(1, int(round(w / 2))), max(1, int(round(h / 2))))
                cv2.ellipse(pupil_fullmax_mask, center, axes, float(angle), 0, 360, 255, thickness=-1)
            pupil_fullmax_d = float(dice_binary_np(pupil_fullmax_mask, gt_pupil[b]))
            mean_fullmax = float(np.mean([lid_d, iris_fullmax_d, pupil_fullmax_d]))

            # ransac_whole: edge → RANSAC
            iris_edge = mask_to_edge(iris_raw, thickness=3)
            pupil_edge = mask_to_edge(pupil_raw, thickness=3)
            iris_params = binary_to_ellipse_params(iris_edge)
            pupil_params = binary_to_ellipse_params(pupil_edge)
            iris_ransac = ellipse_params_to_mask(iris_params, IMAGE_HEIGHT, IMAGE_WIDTH) if iris_params is not None else np.zeros((IMAGE_HEIGHT, IMAGE_WIDTH), np.uint8)
            pupil_ransac = ellipse_params_to_mask(pupil_params, IMAGE_HEIGHT, IMAGE_WIDTH) if pupil_params is not None else np.zeros((IMAGE_HEIGHT, IMAGE_WIDTH), np.uint8)
            iris_ransac_d = float(dice_binary_np(iris_ransac, gt_iris[b]))
            pupil_ransac_d = float(dice_binary_np(pupil_ransac, gt_pupil[b]))
            mean_ransac = float(np.mean([lid_d, iris_ransac_d, pupil_ransac_d]))

            # Primary metric = fullmax
            lid_scores.append(lid_d)
            iris_scores.append(iris_fullmax_d)
            pupil_scores.append(pupil_fullmax_d)

            # per-image rows (long format) with HD95 + NSD
            for mode, i_d, p_d, m_d, i_mask, p_mask in [
                ('raw', iris_raw_d, pupil_raw_d, mean_raw, iris_raw, pupil_raw),
                ('fullmax', iris_fullmax_d, pupil_fullmax_d, mean_fullmax, iris_fullmax_mask, pupil_fullmax_mask),
                ('ransac_whole', iris_ransac_d, pupil_ransac_d, mean_ransac, iris_ransac, pupil_ransac),
            ]:
                row = {
                    'filename': str(filename), 'subject_id': str(subject_id),
                    'mode': mode,
                    'eyelid': lid_d, 'iris': i_d, 'pupil': p_d, 'mean': m_d,
                }
                _augment_hd95_nsd(row, lid_raw, i_mask, p_mask,
                                  gt_lid[b], gt_iris[b], gt_pupil[b])
                per_rows.append(row)

    result = {
        'lid': float(np.mean(lid_scores)) if lid_scores else 0.0,
        'iris': float(np.mean(iris_scores)) if iris_scores else 0.0,
        'pupil': float(np.mean(pupil_scores)) if pupil_scores else 0.0,
        'mean': float(np.mean([np.mean(lid_scores), np.mean(iris_scores), np.mean(pupil_scores)])) if lid_scores else 0.0,
        'iris_before_ellipse': float(np.mean([r['iris'] for r in per_rows if r['mode'] == 'raw'])) if per_rows else 0.0,
        'pupil_before_ellipse': float(np.mean([r['pupil'] for r in per_rows if r['mode'] == 'raw'])) if per_rows else 0.0,
    }
    return result, per_rows



def _boundary_ellipse_fit(pred_labels, region_class, neighbor_class, H, W):
    """Extract contour of region_class that borders neighbor_class, then fitEllipse.

    Args:
        pred_labels: (H, W) predicted class labels
        region_class: class ID of the region to fit (e.g., 2 for iris, 3 for pupil)
        neighbor_class: class ID of the neighbor region (e.g., 1 for conj, 2 for iris)
        H, W: image dimensions

    Returns:
        mask (H, W) uint8 0/255, ellipse params or None
    """
    region_mask = (pred_labels == region_class).astype(np.uint8)
    neighbor_mask = (pred_labels == neighbor_class).astype(np.uint8)

    # Find contour of the region
    contours, _ = cv2.findContours(region_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours:
        return np.zeros((H, W), dtype=np.uint8), None

    # Use the largest contour
    cnt = max(contours, key=cv2.contourArea)

    # Dilate neighbor mask by 2px to catch boundary adjacency
    kernel = np.ones((5, 5), np.uint8)
    neighbor_dilated = cv2.dilate(neighbor_mask, kernel, iterations=1)

    # Filter contour points: keep only those adjacent to neighbor
    boundary_pts = []
    for pt in cnt:
        x, y = pt[0]
        if 0 <= y < H and 0 <= x < W and neighbor_dilated[y, x] > 0:
            boundary_pts.append([x, y])

    if len(boundary_pts) < 5:
        # Fallback: try all contour points
        return np.zeros((H, W), dtype=np.uint8), None

    boundary_pts = np.array(boundary_pts, dtype=np.float32)

    try:
        ellipse = cv2.fitEllipse(boundary_pts)
    except Exception:
        return np.zeros((H, W), dtype=np.uint8), None

    (cx, cy), (w, h), angle = ellipse

    # Sanity check: reject degenerate ellipses
    if w <= 0 or h <= 0 or w > W * 2 or h > H * 2:
        return np.zeros((H, W), dtype=np.uint8), None

    mask = np.zeros((H, W), dtype=np.uint8)
    center = (int(round(cx)), int(round(cy)))
    axes = (max(1, int(round(w / 2))), max(1, int(round(h / 2))))
    cv2.ellipse(mask, center, axes, float(angle), 0, 360, 255, thickness=-1)
    return mask, ellipse


@torch.no_grad()
def evaluate_method6(model, val_loader, device):
    """Method6評価: 4-class visible-only segmentation with boundary-based ellipse fitting.

    Post-processing variants:
    - raw: visible mask union (no ellipse fitting)
    - boundary: iris-conj boundary fit + pupil-iris boundary fit
    - fullmax: largest contour → fitEllipse (standard comparison)

    Returns:
        result_dict (primary=boundary), per_image_rows
    """
    model.eval()
    per_rows = []
    lid_scores, iris_scores, pupil_scores = [], [], []

    for batch in tqdm(val_loader, desc="M6評価", leave=False):
        img = batch['image'].to(device)
        gt_lid   = batch['mask_lid'].cpu().numpy()
        gt_iris  = batch['mask_iris'].cpu().numpy()
        gt_pupil = batch['mask_pupil'].cpu().numpy()
        filenames = batch.get('filename', None)

        with autocast():
            out = model(img)
            logits = out['four_class_seg']  # (B,4,H,W)

        pred_labels = torch.argmax(logits, dim=1).cpu().numpy()  # (B,H,W)

        for b in range(pred_labels.shape[0]):
            pred = pred_labels[b]
            filename = filenames[b] if filenames is not None else str(b)
            subject_id = str(filename).split('-', 1)[0]
            H, W = pred.shape

            # Eyelid: class 1 ∪ 2 ∪ 3 (everything inside eyelid opening)
            lid_pred = (((pred == 1) | (pred == 2) | (pred == 3)).astype(np.uint8) * 255)
            lid_d = float(dice_binary_np(lid_pred, gt_lid[b]))

            # --- raw: visible masks directly (no ellipse) ---
            iris_raw = ((pred == 2).astype(np.uint8) * 255)  # visible iris only
            pupil_raw = ((pred == 3).astype(np.uint8) * 255)  # visible pupil only
            # For raw Dice, compare visible prediction against full GT (amodal)
            iris_raw_d = float(dice_binary_np(iris_raw, gt_iris[b]))
            pupil_raw_d = float(dice_binary_np(pupil_raw, gt_pupil[b]))
            mean_raw = float(np.mean([lid_d, iris_raw_d, pupil_raw_d]))

            # --- boundary: iris-conjunctiva boundary fit, pupil-iris boundary fit ---
            iris_boundary_mask, _ = _boundary_ellipse_fit(pred, region_class=2, neighbor_class=1, H=H, W=W)
            pupil_boundary_mask, _ = _boundary_ellipse_fit(pred, region_class=3, neighbor_class=2, H=H, W=W)
            iris_boundary_d = float(dice_binary_np(iris_boundary_mask, gt_iris[b]))
            pupil_boundary_d = float(dice_binary_np(pupil_boundary_mask, gt_pupil[b]))
            mean_boundary = float(np.mean([lid_d, iris_boundary_d, pupil_boundary_d]))

            # --- fullmax: largest contour → fitEllipse (for comparison) ---
            iris_fullmax_mask = np.zeros((H, W), dtype=np.uint8)
            ell = _fit_ellipse_from_binary(iris_raw)
            if ell is not None:
                (cx, cy), (w, h), angle = ell
                center = (int(round(cx)), int(round(cy)))
                axes = (max(1, int(round(w / 2))), max(1, int(round(h / 2))))
                cv2.ellipse(iris_fullmax_mask, center, axes, float(angle), 0, 360, 255, thickness=-1)
            iris_fullmax_d = float(dice_binary_np(iris_fullmax_mask, gt_iris[b]))

            pupil_fullmax_mask = np.zeros((H, W), dtype=np.uint8)
            ell = _fit_ellipse_from_binary(pupil_raw)
            if ell is not None:
                (cx, cy), (w, h), angle = ell
                center = (int(round(cx)), int(round(cy)))
                axes = (max(1, int(round(w / 2))), max(1, int(round(h / 2))))
                cv2.ellipse(pupil_fullmax_mask, center, axes, float(angle), 0, 360, 255, thickness=-1)
            pupil_fullmax_d = float(dice_binary_np(pupil_fullmax_mask, gt_pupil[b]))
            mean_fullmax = float(np.mean([lid_d, iris_fullmax_d, pupil_fullmax_d]))

            # Primary metric = boundary
            lid_scores.append(lid_d)
            iris_scores.append(iris_boundary_d)
            pupil_scores.append(pupil_boundary_d)

            # per-image rows (long format) with HD95 + NSD
            for mode, i_d, p_d, m_d, i_mask, p_mask in [
                ('raw', iris_raw_d, pupil_raw_d, mean_raw, iris_raw, pupil_raw),
                ('boundary', iris_boundary_d, pupil_boundary_d, mean_boundary, iris_boundary_mask, pupil_boundary_mask),
                ('fullmax', iris_fullmax_d, pupil_fullmax_d, mean_fullmax, iris_fullmax_mask, pupil_fullmax_mask),
            ]:
                row = {
                    'filename': str(filename), 'subject_id': str(subject_id),
                    'mode': mode,
                    'eyelid': lid_d, 'iris': i_d, 'pupil': p_d, 'mean': m_d,
                }
                _augment_hd95_nsd(row, lid_pred, i_mask, p_mask,
                                  gt_lid[b], gt_iris[b], gt_pupil[b])
                per_rows.append(row)

    result = {
        'lid': float(np.mean(lid_scores)) if lid_scores else 0.0,
        'iris': float(np.mean(iris_scores)) if iris_scores else 0.0,
        'pupil': float(np.mean(pupil_scores)) if pupil_scores else 0.0,
        'mean': float(np.mean([np.mean(lid_scores), np.mean(iris_scores), np.mean(pupil_scores)])) if lid_scores else 0.0,
        'iris_before_ellipse': float(np.mean([r['iris'] for r in per_rows if r['mode'] == 'raw'])) if per_rows else 0.0,
        'pupil_before_ellipse': float(np.mean([r['pupil'] for r in per_rows if r['mode'] == 'raw'])) if per_rows else 0.0,
    }
    return result, per_rows



def _boundary_ellipse_fit(pred_labels, region_class, neighbor_class, H, W):
    """Extract contour of region_class that borders neighbor_class, then fitEllipse.

    Args:
        pred_labels: (H, W) predicted class labels
        region_class: class ID of the region to fit (e.g., 2 for iris, 3 for pupil)
        neighbor_class: class ID of the neighbor region (e.g., 1 for conj, 2 for iris)
        H, W: image dimensions

    Returns:
        mask (H, W) uint8 0/255, ellipse params or None
    """
    region_mask = (pred_labels == region_class).astype(np.uint8)
    neighbor_mask = (pred_labels == neighbor_class).astype(np.uint8)

    # Find contour of the region
    contours, _ = cv2.findContours(region_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours:
        return np.zeros((H, W), dtype=np.uint8), None

    # Use the largest contour
    cnt = max(contours, key=cv2.contourArea)

    # Dilate neighbor mask by 2px to catch boundary adjacency
    kernel = np.ones((5, 5), np.uint8)
    neighbor_dilated = cv2.dilate(neighbor_mask, kernel, iterations=1)

    # Filter contour points: keep only those adjacent to neighbor
    boundary_pts = []
    for pt in cnt:
        x, y = pt[0]
        if 0 <= y < H and 0 <= x < W and neighbor_dilated[y, x] > 0:
            boundary_pts.append([x, y])

    if len(boundary_pts) < 5:
        # Fallback: try all contour points
        return np.zeros((H, W), dtype=np.uint8), None

    boundary_pts = np.array(boundary_pts, dtype=np.float32)

    try:
        ellipse = cv2.fitEllipse(boundary_pts)
    except Exception:
        return np.zeros((H, W), dtype=np.uint8), None

    (cx, cy), (w, h), angle = ellipse

    # Sanity check: reject degenerate ellipses
    if w <= 0 or h <= 0 or w > W * 2 or h > H * 2:
        return np.zeros((H, W), dtype=np.uint8), None

    mask = np.zeros((H, W), dtype=np.uint8)
    center = (int(round(cx)), int(round(cy)))
    axes = (max(1, int(round(w / 2))), max(1, int(round(h / 2))))
    cv2.ellipse(mask, center, axes, float(angle), 0, 360, 255, thickness=-1)
    return mask, ellipse


@torch.no_grad()
def evaluate_method6(model, val_loader, device):
    """Method6評価: 4-class visible-only segmentation with boundary-based ellipse fitting.

    Post-processing variants:
    - raw: visible mask union (no ellipse fitting)
    - boundary: iris-conj boundary fit + pupil-iris boundary fit
    - fullmax: largest contour → fitEllipse (standard comparison)

    Returns:
        result_dict (primary=boundary), per_image_rows
    """
    model.eval()
    per_rows = []
    lid_scores, iris_scores, pupil_scores = [], [], []

    for batch in tqdm(val_loader, desc="M6評価", leave=False):
        img = batch['image'].to(device)
        gt_lid   = batch['mask_lid'].cpu().numpy()
        gt_iris  = batch['mask_iris'].cpu().numpy()
        gt_pupil = batch['mask_pupil'].cpu().numpy()
        filenames = batch.get('filename', None)

        with autocast():
            out = model(img)
            logits = out['four_class_seg']  # (B,4,H,W)

        pred_labels = torch.argmax(logits, dim=1).cpu().numpy()  # (B,H,W)

        for b in range(pred_labels.shape[0]):
            pred = pred_labels[b]
            filename = filenames[b] if filenames is not None else str(b)
            subject_id = str(filename).split('-', 1)[0]
            H, W = pred.shape

            # Eyelid: class 1 ∪ 2 ∪ 3 (everything inside eyelid opening)
            lid_pred = (((pred == 1) | (pred == 2) | (pred == 3)).astype(np.uint8) * 255)
            lid_d = float(dice_binary_np(lid_pred, gt_lid[b]))

            # --- raw: visible masks directly (no ellipse) ---
            iris_raw = ((pred == 2).astype(np.uint8) * 255)  # visible iris only
            pupil_raw = ((pred == 3).astype(np.uint8) * 255)  # visible pupil only
            # For raw Dice, compare visible prediction against full GT (amodal)
            iris_raw_d = float(dice_binary_np(iris_raw, gt_iris[b]))
            pupil_raw_d = float(dice_binary_np(pupil_raw, gt_pupil[b]))
            mean_raw = float(np.mean([lid_d, iris_raw_d, pupil_raw_d]))

            # --- boundary: iris-conjunctiva boundary fit, pupil-iris boundary fit ---
            iris_boundary_mask, _ = _boundary_ellipse_fit(pred, region_class=2, neighbor_class=1, H=H, W=W)
            pupil_boundary_mask, _ = _boundary_ellipse_fit(pred, region_class=3, neighbor_class=2, H=H, W=W)
            iris_boundary_d = float(dice_binary_np(iris_boundary_mask, gt_iris[b]))
            pupil_boundary_d = float(dice_binary_np(pupil_boundary_mask, gt_pupil[b]))
            mean_boundary = float(np.mean([lid_d, iris_boundary_d, pupil_boundary_d]))

            # --- fullmax: largest contour → fitEllipse (for comparison) ---
            iris_fullmax_mask = np.zeros((H, W), dtype=np.uint8)
            ell = _fit_ellipse_from_binary(iris_raw)
            if ell is not None:
                (cx, cy), (w, h), angle = ell
                center = (int(round(cx)), int(round(cy)))
                axes = (max(1, int(round(w / 2))), max(1, int(round(h / 2))))
                cv2.ellipse(iris_fullmax_mask, center, axes, float(angle), 0, 360, 255, thickness=-1)
            iris_fullmax_d = float(dice_binary_np(iris_fullmax_mask, gt_iris[b]))

            pupil_fullmax_mask = np.zeros((H, W), dtype=np.uint8)
            ell = _fit_ellipse_from_binary(pupil_raw)
            if ell is not None:
                (cx, cy), (w, h), angle = ell
                center = (int(round(cx)), int(round(cy)))
                axes = (max(1, int(round(w / 2))), max(1, int(round(h / 2))))
                cv2.ellipse(pupil_fullmax_mask, center, axes, float(angle), 0, 360, 255, thickness=-1)
            pupil_fullmax_d = float(dice_binary_np(pupil_fullmax_mask, gt_pupil[b]))
            mean_fullmax = float(np.mean([lid_d, iris_fullmax_d, pupil_fullmax_d]))

            # Primary metric = boundary
            lid_scores.append(lid_d)
            iris_scores.append(iris_boundary_d)
            pupil_scores.append(pupil_boundary_d)

            # per-image rows (long format) with HD95 + NSD
            for mode, i_d, p_d, m_d, i_mask, p_mask in [
                ('raw', iris_raw_d, pupil_raw_d, mean_raw, iris_raw, pupil_raw),
                ('boundary', iris_boundary_d, pupil_boundary_d, mean_boundary, iris_boundary_mask, pupil_boundary_mask),
                ('fullmax', iris_fullmax_d, pupil_fullmax_d, mean_fullmax, iris_fullmax_mask, pupil_fullmax_mask),
            ]:
                row = {
                    'filename': str(filename), 'subject_id': str(subject_id),
                    'mode': mode,
                    'eyelid': lid_d, 'iris': i_d, 'pupil': p_d, 'mean': m_d,
                }
                _augment_hd95_nsd(row, lid_pred, i_mask, p_mask,
                                  gt_lid[b], gt_iris[b], gt_pupil[b])
                per_rows.append(row)

    result = {
        'lid': float(np.mean(lid_scores)) if lid_scores else 0.0,
        'iris': float(np.mean(iris_scores)) if iris_scores else 0.0,
        'pupil': float(np.mean(pupil_scores)) if pupil_scores else 0.0,
        'mean': float(np.mean([np.mean(lid_scores), np.mean(iris_scores), np.mean(pupil_scores)])) if lid_scores else 0.0,
        'iris_before_ellipse': float(np.mean([r['iris'] for r in per_rows if r['mode'] == 'raw'])) if per_rows else 0.0,
        'pupil_before_ellipse': float(np.mean([r['pupil'] for r in per_rows if r['mode'] == 'raw'])) if per_rows else 0.0,
    }
    return result, per_rows


In [ ]:
# ===== 各Foldを評価 =====
print("\n" + "=" * 80)
print("各Fold評価開始")
print("=" * 80)

# --- 実行順依存を排除（評価セル単独実行でも動くように） ---
from pathlib import Path
import json

def evaluate_method4(model, val_loader, device):
    """Method4評価: Method3と同じ出力形式なのでdelegate"""
    return evaluate_method3(model, val_loader, device)

if 'TRAIN_METHODS' not in globals():
    TRAIN_METHODS = [1, 2, 3, 4, 5, 6]

if 'NUM_FOLDS' not in globals():
    NUM_FOLDS = 5

if 'MODEL_DIR' not in globals():
    MODEL_DIR = Path("model/cv_300ep")
    MODEL_DIR.mkdir(parents=True, exist_ok=True)

if 'fold_indices' not in globals():
    with open('fold_indices.json', 'r') as f:
        fold_indices = json.load(f)

if 'image_paths' not in globals():
    IMAGES_DIR = Path("Images/images")
    df = pd.read_csv('image_metadata.csv')
    image_paths = [IMAGES_DIR / row['filename'] for _, row in df.iterrows()]

# Method3の追加比較（raw / outerarc / fullmax / ransac_whole）を per-image で保存
result_dir = Path('results')
result_dir.mkdir(exist_ok=True)
_ts_m3 = datetime.now().strftime('%Y%m%d_%H%M%S')
method3_perimage_csv = result_dir / f"cv_method3_full_vs_exposed_perimage_{_ts_m3}.csv"

method1_rows_all = []
method2_rows_all = []
method3_rows_all = []
method4_rows_all = []
method5_rows_all = []
method6_rows_all = []

evaluation_results = {1: [], 2: [], 3: [], 4: [], 5: [], 6: []}

# === fold-loop skip guard ===
# Skip the heavy fold loop if all per-image CSVs already exist.
# To force a fresh re-evaluation, set FORCE_REEVALUATE = True in a
# preceding cell before running this one.
FORCE_REEVALUATE = globals().get('FORCE_REEVALUATE', False)
_EXPECTED_CSVS = {
    1: 'cv_method1_reloaded_perimage_*.csv',
    2: 'cv_method2_reloaded_perimage_*.csv',
    3: 'cv_method3_full_vs_exposed_perimage_*.csv',
    4: 'cv_method4_full_vs_exposed_perimage_*.csv',
    5: 'cv_method5_amodal_perimage_*.csv',
    6: 'cv_method6_visible_boundary_perimage_*.csv',
}
_PRIMARY_MODE = {
    1: 'ellipse_regression', 2: 'edge_ellipse_fit',
    3: 'fullmax', 4: 'fullmax', 5: 'fullmax', 6: 'boundary',
}

def _latest_final(pattern):
    xs = sorted(p for p in result_dir.glob(pattern) if '_partial' not in p.name)
    return xs[-1] if xs else None

_methods_to_check = [m for m in TRAIN_METHODS if m in _EXPECTED_CSVS]
_existing = {m: _latest_final(_EXPECTED_CSVS[m]) for m in _methods_to_check}
_all_present = all(_existing.get(m) is not None for m in _methods_to_check)

if _all_present and not FORCE_REEVALUATE:
    print("All per-image evaluation CSVs already exist — skipping fold loop.")
    print("(Set FORCE_REEVALUATE = True in a preceding cell to force re-run.)")
    for _m, _path in _existing.items():
        _df = pd.read_csv(_path)
        _pmode = _PRIMARY_MODE[_m]
        for _fold in range(NUM_FOLDS):
            _fdf = _df[(_df['fold'] == _fold) & (_df['mode'] == _pmode)]
            if len(_fdf) == 0:
                continue
            evaluation_results[_m].append({
                'lid':   float(_fdf['eyelid'].mean()),
                'iris':  float(_fdf['iris'].mean()),
                'pupil': float(_fdf['pupil'].mean()),
                'mean':  float(_fdf[['eyelid', 'iris', 'pupil']].mean(axis=1).mean()),
                'fold':  _fold,
                'method': _m,
            })
        print(f"  M{_m}: loaded {_path.name} ({len(_df)} rows)")
else:
    if FORCE_REEVALUATE:
        print("FORCE_REEVALUATE=True — running full fold evaluation loop.")
    else:
        _missing = [m for m in _methods_to_check if _existing.get(m) is None]
        print(f"Missing per-image CSVs for methods {_missing} — running fold loop.")

    for fold_idx in range(NUM_FOLDS):
        print(f"\nFold {fold_idx} 評価中...")

        # Validation dataloader準備
        val_indices = fold_indices[str(fold_idx)]['val']
        val_paths = [image_paths[i] for i in val_indices]
        # キャッシュは古い環境では無いことが多いので、評価では明示的に無効化（推論自体には不要）
        val_ds = EyeSegmentationDataset(
            val_paths, LABEL_SEG_DIR, LABEL_OBB_DIR,
            transform=False,
            use_ellipse_cache=False,
            use_sixcls_direct=True,
        )
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                                num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

        # 各メソッドを評価
        for method_id in TRAIN_METHODS:
            model_path = MODEL_DIR / f"method{method_id}_fold{fold_idx}_best.pth"
            if not model_path.exists():
                print(f"  ⚠️ Method{method_id} モデルが見つかりません: {model_path}")
                continue

            # モデルロード
            if method_id == 1:
                model = UNetMethod1().to(device)
            elif method_id == 2:
                model = UNetMethod2().to(device)
            elif method_id == 3:
                model = UNetMethod3().to(device)
            elif method_id == 4:
                model = UNetMethod4().to(device)
            elif method_id == 5:
                model = UNetMethod5().to(device)
            else:  # method_id == 6
                model = UNetMethod6().to(device)

            checkpoint = torch.load(model_path, map_location=device)
            model.load_state_dict(checkpoint['model'])
            model.eval()

            # 評価実行
            if method_id == 1:
                result, per_rows = evaluate_method1(model, val_loader, device)
                for r in per_rows: r['fold'] = fold_idx
                method1_rows_all.extend(per_rows)
            elif method_id == 2:
                result, per_rows = evaluate_method2(model, val_loader, device)
                for r in per_rows: r['fold'] = fold_idx
                method2_rows_all.extend(per_rows)
            elif method_id == 3:
                result, per_rows = evaluate_method3(model, val_loader, device)
                # fold列付与して蓄積
                for r in per_rows:
                    r['fold'] = fold_idx
                method3_rows_all.extend(per_rows)
            elif method_id == 4:
                result, per_rows = evaluate_method4(model, val_loader, device)
                for r in per_rows:
                    r['fold'] = fold_idx
                method4_rows_all.extend(per_rows)
            elif method_id == 5:
                result, per_rows = evaluate_method5(model, val_loader, device)
                for r in per_rows:
                    r['fold'] = fold_idx
                method5_rows_all.extend(per_rows)
            else:  # method_id == 6
                result, per_rows = evaluate_method6(model, val_loader, device)
                for r in per_rows:
                    r['fold'] = fold_idx
                method6_rows_all.extend(per_rows)

            result['fold'] = fold_idx
            result['method'] = method_id
            evaluation_results[method_id].append(result)

            print(f"  Method{method_id} | 平均Dice: {result['mean']:.4f}")

            del model
            torch.cuda.empty_cache()

        del val_loader, val_ds
        torch.cuda.empty_cache()

        # === incremental fold save ===
        # Save per-image rows so far as "*_partial.csv" after each fold completes.
        # The partial files are overwritten each fold with the cumulative contents;
        # if the run is interrupted, the last-written partial CSV is the state at
        # the end of the previous fold.
        for _m_id, _rows, _stem in [
            (1, method1_rows_all, 'cv_method1_reloaded_perimage'),
            (2, method2_rows_all, 'cv_method2_reloaded_perimage'),
            (3, method3_rows_all, 'cv_method3_full_vs_exposed_perimage'),
            (4, method4_rows_all, 'cv_method4_full_vs_exposed_perimage'),
            (5, method5_rows_all, 'cv_method5_amodal_perimage'),
            (6, method6_rows_all, 'cv_method6_visible_boundary_perimage'),
        ]:
            if len(_rows) > 0:
                pd.DataFrame(_rows).to_csv(result_dir / f'{_stem}_partial.csv', index=False)
        print(f"  📝 Partial CSVs saved through fold {fold_idx} (n_rows: "
              f"M1={len(method1_rows_all)} M2={len(method2_rows_all)} M3={len(method3_rows_all)} "
              f"M4={len(method4_rows_all)} M5={len(method5_rows_all)} M6={len(method6_rows_all)})")

# per-image CSV保存
if len(method1_rows_all) > 0:
    _ts_m1 = datetime.now().strftime("%Y%m%d_%H%M%S")
    method1_perimage_csv = result_dir / f"cv_method1_reloaded_perimage_{_ts_m1}.csv"
    pd.DataFrame(method1_rows_all).to_csv(method1_perimage_csv, index=False)
    print(f"\n✅ Method1 per-image (DSC+HD95+NSD) saved: {method1_perimage_csv}")

if len(method2_rows_all) > 0:
    _ts_m2 = datetime.now().strftime("%Y%m%d_%H%M%S")
    method2_perimage_csv = result_dir / f"cv_method2_reloaded_perimage_{_ts_m2}.csv"
    pd.DataFrame(method2_rows_all).to_csv(method2_perimage_csv, index=False)
    print(f"\n✅ Method2 per-image (DSC+HD95+NSD) saved: {method2_perimage_csv}")

if len(method3_rows_all) > 0:
    df_m3 = pd.DataFrame(method3_rows_all)
    df_m3.to_csv(method3_perimage_csv, index=False)
    print(f"\n✅ Method3 per-image（full vs 露出）保存: {method3_perimage_csv}")

if len(method4_rows_all) > 0:
    _ts_m4 = datetime.now().strftime("%Y%m%d_%H%M%S")
    method4_perimage_csv = result_dir / f"cv_method4_full_vs_exposed_perimage_{_ts_m4}.csv"
    df_m4 = pd.DataFrame(method4_rows_all)
    df_m4.to_csv(method4_perimage_csv, index=False)
    print(f"\n✅ Method4 per-image（full vs 露出）保存: {method4_perimage_csv}")

if len(method5_rows_all) > 0:
    _ts_m5 = datetime.now().strftime("%Y%m%d_%H%M%S")
    method5_perimage_csv = result_dir / f"cv_method5_amodal_perimage_{_ts_m5}.csv"
    df_m5 = pd.DataFrame(method5_rows_all)
    df_m5.to_csv(method5_perimage_csv, index=False)
    print(f"\n✅ Method5 per-image（amodal）保存: {method5_perimage_csv}")


if len(method6_rows_all) > 0:
    _ts_m6 = datetime.now().strftime("%Y%m%d_%H%M%S")
    method6_perimage_csv = result_dir / f"cv_method6_visible_boundary_perimage_{_ts_m6}.csv"
    df_m6 = pd.DataFrame(method6_rows_all)
    df_m6.to_csv(method6_perimage_csv, index=False)
    print(f"\n✅ Method6 per-image（visible-only boundary fit）保存: {method6_perimage_csv}")


if len(method6_rows_all) > 0:
    _ts_m6 = datetime.now().strftime("%Y%m%d_%H%M%S")
    method6_perimage_csv = result_dir / f"cv_method6_visible_boundary_perimage_{_ts_m6}.csv"
    df_m6 = pd.DataFrame(method6_rows_all)
    df_m6.to_csv(method6_perimage_csv, index=False)
    print(f"\n✅ Method6 per-image（visible-only boundary fit）保存: {method6_perimage_csv}")

print("\n" + "=" * 80)
print("✅ 全Fold評価完了")
print("=" * 80)

## 12. 結果集計・保存


## 12. 結果サマリー

In [11]:
# メソッド別サマリー保存
summary_rows = []
for method_id in TRAIN_METHODS:
    if method_id not in evaluation_results or len(evaluation_results[method_id]) == 0:
        continue
    lids = [r['lid'] for r in evaluation_results[method_id]]
    irises = [r['iris'] for r in evaluation_results[method_id]]
    pupils = [r['pupil'] for r in evaluation_results[method_id]]
    means = [r['mean'] for r in evaluation_results[method_id]]
    summary_rows.append({
        'Method': f"Method{method_id}",
        'Eyelid_mean': float(np.mean(lids)),
        'Eyelid_std': float(np.std(lids)),
        'Iris_mean': float(np.mean(irises)),
        'Iris_std': float(np.std(irises)),
        'Pupil_mean': float(np.mean(pupils)),
        'Pupil_std': float(np.std(pupils)),
        'Total_mean': float(np.mean(means)),
        'Total_std': float(np.std(means))
    })

# ===== 結果をCSVで保存 =====
result_dir = Path("results")
result_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

summary_df = pd.DataFrame(summary_rows) if summary_rows else None
if summary_df is not None:
    print("\n【メソッド別 平均Diceまとめ】")
    print(summary_df.to_string(index=False))
    summary_csv = result_dir / f"cv_eval_summary_{timestamp}.csv"
    summary_df.to_csv(summary_csv, index=False)
    print(f"✅ サマリー保存: {summary_csv}")




【メソッド別 平均Diceまとめ】
 Method  Eyelid_mean  Eyelid_std  Iris_mean  Iris_std  Pupil_mean  Pupil_std  Total_mean  Total_std
Method1     0.984463    0.002228   0.887865  0.003130    0.722505   0.037343    0.864945   0.012092
Method2     0.951272    0.013221   0.895809  0.010736    0.885866   0.023028    0.910982   0.013799
Method3     0.980388    0.002217   0.934587  0.008513    0.902430   0.012370    0.939135   0.006037
Method4     0.983280    0.002182   0.957704  0.005718    0.916146   0.009845    0.952376   0.004398
Method5     0.982455    0.001418   0.958364  0.006203    0.933891   0.007833    0.958236   0.003942
Method6     0.985559    0.002060   0.939329  0.009267    0.926779   0.009995    0.950556   0.005029
✅ サマリー保存: results\cv_eval_summary_20260325_093809.csv


In [12]:
print("\n" + "=" * 80)
print("Cross-Validation 結果サマリー")
print("=" * 80)

# --- 実行順依存を排除（評価だけ回した場合でも落ちないように） ---
from pathlib import Path
import glob

if 'TRAIN_METHODS' not in globals():
    TRAIN_METHODS = [1, 2, 3, 4, 5, 6]

# fold_results が無い場合は、保存済みの学習CSVから復元（無ければ学習結果セクションをスキップ）
if 'fold_results' not in globals():
    fold_results = {m: [] for m in TRAIN_METHODS}
    result_dir = Path('results')
    for m in TRAIN_METHODS:
        files = sorted(result_dir.glob(f"cv_train_method{m}_*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
        if not files:
            continue
        p = files[0]
        try:
            df_tmp = pd.read_csv(p)
            fold_results[m] = df_tmp.to_dict('records')
            print(f"ℹ️ fold_results を復元: Method{m} ← {p.resolve()}")
        except Exception as e:
            print(f"⚠️ Method{m} 学習CSVの読み込みに失敗: {p} ({e})")

# ===== 学習結果 =====
print("\n【学習結果 - Best Validation Loss】")
_any_train = False
for method_id in TRAIN_METHODS:
    if method_id not in fold_results or len(fold_results[method_id]) == 0:
        continue
    _any_train = True
    print(f"\n== Method {method_id} ==")
    train_df = pd.DataFrame(fold_results[method_id])
    if 'best_val_loss' in train_df.columns:
        print(train_df[['fold', 'best_val_loss']].to_string(index=False))
        print(f"平均: {train_df['best_val_loss'].mean():.4f} ± {train_df['best_val_loss'].std():.4f}")
    else:
        # 旧CSVなどで列が違う場合
        print(train_df.head().to_string(index=False))

if not _any_train:
    print("⚠️ fold_results（学習結果）が見つかりません。学習セルを実行するか、results/cv_train_method*.csv を確認してください。")

# ===== 評価結果 =====
print("\n【評価結果 - Dice係数】")

# Method1とMethod2: Eyelid, Iris, Pupilの3つ
for method_id in [1, 2]:
    if method_id not in evaluation_results or len(evaluation_results[method_id]) == 0:
        continue
    print(f"\n== Method {method_id} ==")
    eval_data = []
    for result in evaluation_results[method_id]:
        row = {
            'Fold': result['fold'],
            'Eyelid': result['lid'],
            'Iris': result['iris'],
            'Pupil': result['pupil'],
            'Mean': result['mean']
        }
        eval_data.append(row)
    
    eval_df = pd.DataFrame(eval_data)
    print(eval_df.to_string(index=False))
    
    # 平均と標準偏差
    lid_scores = [r['lid'] for r in evaluation_results[method_id]]
    iris_scores = [r['iris'] for r in evaluation_results[method_id]]
    pupil_scores = [r['pupil'] for r in evaluation_results[method_id]]
    mean_scores = [r['mean'] for r in evaluation_results[method_id]]
    
    print(f"\n平均 ± 標準偏差:")
    print(f"  Eyelid: {np.mean(lid_scores):.4f} ± {np.std(lid_scores):.4f}")
    print(f"  Iris:   {np.mean(iris_scores):.4f} ± {np.std(iris_scores):.4f}")
    print(f"  Pupil:  {np.mean(pupil_scores):.4f} ± {np.std(pupil_scores):.4f}")
    print(f"  Mean:   {np.mean(mean_scores):.4f} ± {np.std(mean_scores):.4f}")

# Method3: Eyelid, Iris, Pupilの3つ + 参考値（楕円近似前）
if 3 in evaluation_results and len(evaluation_results[3]) > 0:
    print(f"\n== Method 3 ==")
    eval_data = []
    for result in evaluation_results[3]:
        row = {
            'Fold': result['fold'],
            'Eyelid': result['lid'],
            'Iris': result['iris'],
            'Pupil': result['pupil'],
            'Mean': result['mean']
        }
        eval_data.append(row)
    
    eval_df = pd.DataFrame(eval_data)
    print(eval_df.to_string(index=False))
    
    # 平均と標準偏差（楕円近似後）
    lid_scores = [r['lid'] for r in evaluation_results[3]]
    iris_scores = [r['iris'] for r in evaluation_results[3]]
    pupil_scores = [r['pupil'] for r in evaluation_results[3]]
    mean_scores = [r['mean'] for r in evaluation_results[3]]
    
    print(f"\n平均 ± 標準偏差（楕円近似後）:")
    print(f"  Eyelid: {np.mean(lid_scores):.4f} ± {np.std(lid_scores):.4f}")
    print(f"  Iris:   {np.mean(iris_scores):.4f} ± {np.std(iris_scores):.4f}")
    print(f"  Pupil:  {np.mean(pupil_scores):.4f} ± {np.std(pupil_scores):.4f}")
    print(f"  Mean:   {np.mean(mean_scores):.4f} ± {np.std(mean_scores):.4f}")
    
    # 参考値：楕円近似前
    if 'iris_before_ellipse' in evaluation_results[3][0] and 'pupil_before_ellipse' in evaluation_results[3][0]:
        iris_before = [r['iris_before_ellipse'] for r in evaluation_results[3]]
        pupil_before = [r['pupil_before_ellipse'] for r in evaluation_results[3]]
        print(f"\n参考値（楕円近似前 - 6クラス→合成マスク）:")
        print(f"  Iris:   {np.mean(iris_before):.4f} ± {np.std(iris_before):.4f}")
        print(f"  Pupil:  {np.mean(pupil_before):.4f} ± {np.std(pupil_before):.4f}")

# Method4: Method3と同形式
if 4 in evaluation_results and len(evaluation_results[4]) > 0:
    print(f"\n== Method 4 (CE+Dice) ==")
    eval_data = []
    for result in evaluation_results[4]:
        row = {
            'Fold': result['fold'],
            'Eyelid': result['lid'],
            'Iris': result['iris'],
            'Pupil': result['pupil'],
            'Mean': result['mean']
        }
        eval_data.append(row)

    eval_df = pd.DataFrame(eval_data)
    print(eval_df.to_string(index=False))

    lid_scores = [r['lid'] for r in evaluation_results[4]]
    iris_scores = [r['iris'] for r in evaluation_results[4]]
    pupil_scores = [r['pupil'] for r in evaluation_results[4]]
    mean_scores = [r['mean'] for r in evaluation_results[4]]

    print(f"\n平均 ± 標準偏差（楕円近似後）:")
    print(f"  Eyelid: {np.mean(lid_scores):.4f} ± {np.std(lid_scores):.4f}")
    print(f"  Iris:   {np.mean(iris_scores):.4f} ± {np.std(iris_scores):.4f}")
    print(f"  Pupil:  {np.mean(pupil_scores):.4f} ± {np.std(pupil_scores):.4f}")
    print(f"  Mean:   {np.mean(mean_scores):.4f} ± {np.std(mean_scores):.4f}")

    if 'iris_before_ellipse' in evaluation_results[4][0] and 'pupil_before_ellipse' in evaluation_results[4][0]:
        iris_before = [r['iris_before_ellipse'] for r in evaluation_results[4]]
        pupil_before = [r['pupil_before_ellipse'] for r in evaluation_results[4]]
        print(f"\n参考値（楕円近似前 - 6クラス→合成マスク）:")
        print(f"  Iris:   {np.mean(iris_before):.4f} ± {np.std(iris_before):.4f}")
        print(f"  Pupil:  {np.mean(pupil_before):.4f} ± {np.std(pupil_before):.4f}")


# Method5: 3-class amodal
if 5 in evaluation_results and len(evaluation_results[5]) > 0:
    print(f"\n== Method 5 (3-class Amodal) ==")
    eval_data = []
    for result in evaluation_results[5]:
        row = {
            'Fold': result['fold'],
            'Eyelid': result['lid'],
            'Iris': result['iris'],
            'Pupil': result['pupil'],
            'Mean': result['mean']
        }
        eval_data.append(row)

    eval_df = pd.DataFrame(eval_data)
    print(eval_df.to_string(index=False))

    lid_scores = [r['lid'] for r in evaluation_results[5]]
    iris_scores = [r['iris'] for r in evaluation_results[5]]
    pupil_scores = [r['pupil'] for r in evaluation_results[5]]
    mean_scores = [r['mean'] for r in evaluation_results[5]]

    print(f"\n平均 ± 標準偏差（楕円近似後）:")
    print(f"  Eyelid: {np.mean(lid_scores):.4f} ± {np.std(lid_scores):.4f}")
    print(f"  Iris:   {np.mean(iris_scores):.4f} ± {np.std(iris_scores):.4f}")
    print(f"  Pupil:  {np.mean(pupil_scores):.4f} ± {np.std(pupil_scores):.4f}")
    print(f"  Mean:   {np.mean(mean_scores):.4f} ± {np.std(mean_scores):.4f}")

    if 'iris_before_ellipse' in evaluation_results[5][0] and 'pupil_before_ellipse' in evaluation_results[5][0]:
        iris_before = [r['iris_before_ellipse'] for r in evaluation_results[5]]
        pupil_before = [r['pupil_before_ellipse'] for r in evaluation_results[5]]
        print(f"\n参考値（楕円近似前 - raw sigmoid mask）:")
        print(f"  Iris:   {np.mean(iris_before):.4f} ± {np.std(iris_before):.4f}")
        print(f"  Pupil:  {np.mean(pupil_before):.4f} ± {np.std(pupil_before):.4f}")


# Method6: 4-class visible-only (boundary-based ellipse fitting)
if 6 in evaluation_results and len(evaluation_results[6]) > 0:
    print(f"\n== Method 6 (4-class Visible-only, Boundary Fit) ==")
    eval_data = []
    for result in evaluation_results[6]:
        row = {
            'Fold': result['fold'],
            'Eyelid': result['lid'],
            'Iris': result['iris'],
            'Pupil': result['pupil'],
            'Mean': result['mean']
        }
        eval_data.append(row)

    eval_df = pd.DataFrame(eval_data)
    print(eval_df.to_string(index=False))

    lid_scores = [r['lid'] for r in evaluation_results[6]]
    iris_scores = [r['iris'] for r in evaluation_results[6]]
    pupil_scores = [r['pupil'] for r in evaluation_results[6]]
    mean_scores = [r['mean'] for r in evaluation_results[6]]

    print(f"\n平均 ± 標準偏差（境界ベース楕円近似後）:")
    print(f"  Eyelid: {np.mean(lid_scores):.4f} ± {np.std(lid_scores):.4f}")
    print(f"  Iris:   {np.mean(iris_scores):.4f} ± {np.std(iris_scores):.4f}")
    print(f"  Pupil:  {np.mean(pupil_scores):.4f} ± {np.std(pupil_scores):.4f}")
    print(f"  Mean:   {np.mean(mean_scores):.4f} ± {np.std(mean_scores):.4f}")

    if 'iris_before_ellipse' in evaluation_results[6][0] and 'pupil_before_ellipse' in evaluation_results[6][0]:
        iris_before = [r['iris_before_ellipse'] for r in evaluation_results[6]]
        pupil_before = [r['pupil_before_ellipse'] for r in evaluation_results[6]]
        print(f"\n参考値（楕円近似前 - visible mask）:")
        print(f"  Iris:   {np.mean(iris_before):.4f} ± {np.std(iris_before):.4f}")
        print(f"  Pupil:  {np.mean(pupil_before):.4f} ± {np.std(pupil_before):.4f}")


# Method6: 4-class visible-only (boundary-based ellipse fitting)
if 6 in evaluation_results and len(evaluation_results[6]) > 0:
    print(f"\n== Method 6 (4-class Visible-only, Boundary Fit) ==")
    eval_data = []
    for result in evaluation_results[6]:
        row = {
            'Fold': result['fold'],
            'Eyelid': result['lid'],
            'Iris': result['iris'],
            'Pupil': result['pupil'],
            'Mean': result['mean']
        }
        eval_data.append(row)

    eval_df = pd.DataFrame(eval_data)
    print(eval_df.to_string(index=False))

    lid_scores = [r['lid'] for r in evaluation_results[6]]
    iris_scores = [r['iris'] for r in evaluation_results[6]]
    pupil_scores = [r['pupil'] for r in evaluation_results[6]]
    mean_scores = [r['mean'] for r in evaluation_results[6]]

    print(f"\n平均 ± 標準偏差（境界ベース楕円近似後）:")
    print(f"  Eyelid: {np.mean(lid_scores):.4f} ± {np.std(lid_scores):.4f}")
    print(f"  Iris:   {np.mean(iris_scores):.4f} ± {np.std(iris_scores):.4f}")
    print(f"  Pupil:  {np.mean(pupil_scores):.4f} ± {np.std(pupil_scores):.4f}")
    print(f"  Mean:   {np.mean(mean_scores):.4f} ± {np.std(mean_scores):.4f}")

    if 'iris_before_ellipse' in evaluation_results[6][0] and 'pupil_before_ellipse' in evaluation_results[6][0]:
        iris_before = [r['iris_before_ellipse'] for r in evaluation_results[6]]
        pupil_before = [r['pupil_before_ellipse'] for r in evaluation_results[6]]
        print(f"\n参考値（楕円近似前 - visible mask）:")
        print(f"  Iris:   {np.mean(iris_before):.4f} ± {np.std(iris_before):.4f}")
        print(f"  Pupil:  {np.mean(pupil_before):.4f} ± {np.std(pupil_before):.4f}")

# ===== 手法比較 =====
print("\n【手法比較】")
comparison_data = []
for method_id in TRAIN_METHODS:
    if method_id in evaluation_results and len(evaluation_results[method_id]) > 0:
        mean_scores = [r['mean'] for r in evaluation_results[method_id]]
        comparison_data.append({
            'Method': f"Method{method_id}",
            'Mean Dice': np.mean(mean_scores),
            'Std': np.std(mean_scores)
        })

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

# ===== 結果をCSVで保存 =====
result_dir = Path("results")
result_dir.mkdir(exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# 学習結果
for method_id in TRAIN_METHODS:
    train_csv = result_dir / f"cv_train_method{method_id}_{timestamp}.csv"
    train_df = pd.DataFrame(fold_results[method_id])
    train_df.to_csv(train_csv, index=False)
    print(f"\n✅ Method{method_id} 学習結果保存: {train_csv}")

# 評価結果
for method_id in TRAIN_METHODS:
    if method_id in evaluation_results and len(evaluation_results[method_id]) > 0:
        eval_csv = result_dir / f"cv_eval_method{method_id}_{timestamp}.csv"
        
        if method_id in [1, 2]:
            eval_data = []
            for result in evaluation_results[method_id]:
                row = {
                    'fold': result['fold'],
                    'eyelid': result['lid'],
                    'iris': result['iris'],
                    'pupil': result['pupil'],
                    'mean': result['mean']
                }
                eval_data.append(row)
            eval_df = pd.DataFrame(eval_data)
        else:  # method_id in [3, 4, 5]
            eval_data = []
            for result in evaluation_results[method_id]:
                row = {
                    'fold': result['fold'],
                    'eyelid': result['lid'],
                    'iris': result['iris'],
                    'pupil': result['pupil'],
                    'mean': result['mean']
                }
                # 参考値（存在する場合のみ）
                if 'iris_before_ellipse' in result and 'pupil_before_ellipse' in result:
                    row['iris_before_ellipse'] = result['iris_before_ellipse']
                    row['pupil_before_ellipse'] = result['pupil_before_ellipse']
                eval_data.append(row)
            eval_df = pd.DataFrame(eval_data)
        
        eval_df.to_csv(eval_csv, index=False)
        print(f"✅ Method{method_id} 評価結果保存: {eval_csv}")

# 比較結果
comparison_csv = result_dir / f"cv_comparison_{timestamp}.csv"
comparison_df.to_csv(comparison_csv, index=False)
print(f"✅ 比較結果保存: {comparison_csv}")

print("\n" + "=" * 80)
print("✅ 5-Fold Cross-Validation 完了（Method1/2/3/4/5/6）")
print("=" * 80)



Cross-Validation 結果サマリー

【学習結果 - Best Validation Loss】

== Method 1 ==
 fold  best_val_loss
    0       0.085662
    1       0.065349
    2       0.086572
    3       0.087554
    4       0.096698
平均: 0.0844 ± 0.0115

== Method 2 ==
 fold  best_val_loss
    0       0.014557
    1       0.013635
    2       0.014779
    3       0.014715
    4       0.017787
平均: 0.0151 ± 0.0016

== Method 3 ==
 fold  best_val_loss
    0       0.104532
    1       0.113355
    2       0.107603
    3       0.109065
    4       0.098280
平均: 0.1066 ± 0.0056

== Method 4 ==
 fold  best_val_loss
    0       0.084750
    1       0.093832
    2       0.090741
    3       0.090565
    4       0.084296
平均: 0.0888 ± 0.0041

== Method 5 ==
 fold  best_val_loss
    0       0.052784
    1       0.062959
    2       0.061093
    3       0.068761
    4       0.055298
平均: 0.0602 ± 0.0063

== Method 6 ==
 fold  best_val_loss
    0       0.031527
    1       0.030635
    2       0.033549
    3       0.037642
    4       0

In [13]:
# ===== Method3（U-Net）: full vs 露出 の被験者クラスタ統計（推奨） =====
# 入力: results/cv_method3_full_vs_exposed_perimage_*.csv

import numpy as np
import pandas as pd
from pathlib import Path

result_dir = Path('results')
files = sorted(result_dir.glob('cv_method3_full_vs_exposed_perimage_*.csv'), key=lambda p: p.stat().st_mtime, reverse=True)
if not files:
    print('⚠️ per-image CSVが見つかりません。先に評価セル（各Fold評価）を実行してください。')
else:
    per_csv = files[0]
    print(f"✅ 入力CSV: {per_csv.resolve()}")
    df = pd.read_csv(per_csv)

    # subject_idがなければ filenameから復元
    if 'subject_id' not in df.columns:
        df['subject_id'] = df['filename'].astype(str).str.split('-', n=1).str[0]

    # 被験者×mode で平均
    metric_cols = ['mean','eyelid','iris','pupil']
    subj = df.groupby(['subject_id','mode'])[metric_cols].mean().reset_index()

    # wide化
    wide = subj.pivot(index='subject_id', columns='mode', values='mean')

    def holm_adjust(pvals: dict):
        items = sorted(pvals.items(), key=lambda kv: kv[1])
        m = len(items)
        out = {}
        prev = 0.0
        for j, (name, p) in enumerate(items, start=1):
            adj = (m - j + 1) * p
            adj = min(1.0, adj)
            adj = max(prev, adj)
            prev = adj
            out[name] = adj
        return out

    def paired_cluster_permutation_pvalue(diff: np.ndarray, n_perm=20000, seed=0):
        rng = np.random.default_rng(seed)
        obs = float(np.mean(diff))
        signs = rng.choice(np.array([-1.0, 1.0]), size=(n_perm, diff.size), replace=True)
        perm = (signs * diff[None, :]).mean(axis=1)
        p = float((np.abs(perm) >= abs(obs)).mean())
        return obs, p

    def paired_cluster_bootstrap_ci(diff: np.ndarray, n_boot=20000, seed=1):
        rng = np.random.default_rng(seed)
        n = diff.size
        idx = rng.integers(0, n, size=(n_boot, n))
        boot = diff[idx].mean(axis=1)
        lo, hi = np.quantile(boot, [0.025, 0.975])
        return float(lo), float(hi)

    # ranking
    modes = [c for c in wide.columns if c in ['raw','outerarc','fullmax','ransac_whole']]
    rank_rows = []
    for m in modes:
        rank_rows.append({'method': m, 'subject_mean_dice': float(np.nanmean(wide[m])), 'n_subjects': int(wide[m].notna().sum())})
    df_rank = pd.DataFrame(rank_rows).sort_values('subject_mean_dice', ascending=False)
    print('\n--- Method3: 被験者平均Dice（mean）のランキング ---')
    print(df_rank.to_string(index=False))

    # direct comparisons (full vs exposed)
    print('\n--- 直接比較（mean Dice）：full vs 露出（被験者クラスタ / 置換検定 + Holm） ---')
    pair_specs = [
        ('fullmax','outerarc'),
        ('ransac_whole','raw'),  # 参考: 既存Method3(RANSAC) vs raw
    ]
    rows = []
    pvals = {}
    for a,b in pair_specs:
        if a not in wide.columns or b not in wide.columns:
            continue
        common = wide[[a,b]].dropna().index
        if len(common) < 5:
            continue
        diff = (wide.loc[common, a] - wide.loc[common, b]).to_numpy(dtype=float)
        obs,p = paired_cluster_permutation_pvalue(diff, n_perm=20000, seed=10)
        lo,hi = paired_cluster_bootstrap_ci(diff, n_boot=20000, seed=11)
        win = float((diff>0).mean())
        key = f"{a}-{b}"
        rows.append({'compare': f"{a} - {b}", 'n_subjects': int(len(common)), 'mean_diff': float(obs), 'ci95_low': float(lo), 'ci95_high': float(hi), 'win_rate': float(win), 'p_perm': float(p)})
        pvals[key] = p

    if rows:
        p_holm = holm_adjust(pvals)
        for r in rows:
            key = r['compare'].replace(' - ','-')
            r['p_holm'] = float(p_holm.get(key, np.nan))
        df_pair = pd.DataFrame(rows).sort_values('p_perm')
        print(df_pair.to_string(index=False))
    else:
        print('⚠️ 比較に必要なmode列が不足しています（per-image CSVを確認してください）。')



✅ 入力CSV: C:\Users\CorneAI\Eyelid_Iris_pupil_seg_comparison\results\cv_method3_full_vs_exposed_perimage_20260324_141218.csv

--- Method3: 被験者平均Dice（mean）のランキング ---
      method  subject_mean_dice  n_subjects
     fullmax           0.939232         122
ransac_whole           0.934312         122
    outerarc           0.930788         122
         raw           0.917460         122

--- 直接比較（mean Dice）：full vs 露出（被験者クラスタ / 置換検定 + Holm） ---
           compare  n_subjects  mean_diff  ci95_low  ci95_high  win_rate  p_perm  p_holm
fullmax - outerarc         122   0.008443  0.005365   0.012364  0.836066     0.0     0.0
ransac_whole - raw         122   0.016851  0.014677   0.019125  0.975410     0.0     0.0


## 13. 可視化（オプション）


In [ ]:
import matplotlib.pyplot as plt

# クラスカラーマップ（BGRではなくRGB）
CLASS_COLORS_RGB = {
    0: [0, 0, 0],           # 背景 - 黒
    1: [0, 0, 255],         # lid（まぶた） - 青
    2: [0, 255, 0],         # iris_vis（可視虹彩） - 緑
    3: [255, 0, 0],         # iris_occ（遮蔽虹彩） - 赤
    4: [255, 255, 0],       # pupil_vis（可視瞳孔） - 黄色
    5: [255, 0, 255],       # pupil_occ（遮蔽瞳孔） - マゼンタ
}

def create_colored_segmentation(labels):
    """6クラスのラベルマップをRGB画像に変換"""
    H, W = labels.shape
    colored = np.zeros((H, W, 3), dtype=np.uint8)
    
    for class_id, color in CLASS_COLORS_RGB.items():
        mask = labels == class_id
        colored[mask] = color
    
    return colored

def denorm_img(t):
    """正規化を戻す"""
    t = t.clone().cpu()
    mean = torch.tensor([0.485, 0.456, 0.406])[:,None,None]
    std  = torch.tensor([0.229, 0.224, 0.225])[:,None,None]
    t = t*std + mean
    return np.clip(t.permute(1,2,0).numpy(), 0, 1)

def visualize_all_methods(sample, models, device):
    """全メソッドを比較可視化（Original | M1 | M2 | M3 | M4）"""
    img_t = sample['image'].unsqueeze(0).to(device)
    img_vis = denorm_img(sample['image'])
    
    # 可視化用画像（Eyelid, Iris, Pupilの3行）
    fig, axes = plt.subplots(3, 5, figsize=(20, 12))
    
    # 列タイトル
    col_titles = ['Original', 'Method1', 'Method2', 'Method3', 'Method4']
    row_labels = ['Eyelid', 'Iris', 'Pupil']
    
    # GTマスク
    gt_lid = sample['mask_lid'].numpy()
    gt_iris = sample['mask_iris'].numpy()
    gt_pupil = sample['mask_pupil'].numpy()
    
    # === Method1の予測 ===
    if 1 in models and models[1] is not None:
        models[1].eval()
        with torch.no_grad(), autocast():
            out1 = models[1](img_t)
        
        lid1 = (torch.sigmoid(out1['eyelid_seg'][0:1]).cpu().squeeze().numpy() >= 0.5).astype(np.uint8)*255
        iris_params = torch.sigmoid(out1['iris_ellipse']).cpu().numpy()[0]
        pupil_params = torch.sigmoid(out1['pupil_ellipse']).cpu().numpy()[0]
        iris1 = ellipse_params_to_mask(iris_params, IMAGE_HEIGHT, IMAGE_WIDTH)
        pupil1 = ellipse_params_to_mask(pupil_params, IMAGE_HEIGHT, IMAGE_WIDTH)
    else:
        lid1 = iris1 = pupil1 = np.zeros((IMAGE_HEIGHT, IMAGE_WIDTH), np.uint8)
    
    # === Method2の予測 ===
    if 2 in models and models[2] is not None:
        models[2].eval()
        with torch.no_grad(), autocast():
            out2 = models[2](img_t)
            edge_logits = out2['edge_logits']
        
        lid_edge = (torch.sigmoid(edge_logits[0,0:1]).cpu().squeeze().numpy() >= 0.5).astype(np.uint8)*255
        lid2 = bin_edge_to_filled(lid_edge)
        
        iris_edge = (torch.sigmoid(edge_logits[0,1:2]).cpu().squeeze().numpy() >= 0.5).astype(np.uint8)*255
        pupil_edge = (torch.sigmoid(edge_logits[0,2:3]).cpu().squeeze().numpy() >= 0.5).astype(np.uint8)*255
        
        iris2 = np.zeros((IMAGE_HEIGHT, IMAGE_WIDTH), np.uint8)
        pupil2 = np.zeros((IMAGE_HEIGHT, IMAGE_WIDTH), np.uint8)
        
        iris_pts = np.column_stack(np.where(iris_edge > 0))
        if len(iris_pts) >= 5:
            try:
                ellipse = cv2.fitEllipse(iris_pts[:, ::-1].astype(np.int32))
                cv2.ellipse(iris2, ellipse, 255, thickness=-1)
            except: pass
        
        pupil_pts = np.column_stack(np.where(pupil_edge > 0))
        if len(pupil_pts) >= 5:
            try:
                ellipse = cv2.fitEllipse(pupil_pts[:, ::-1].astype(np.int32))
                cv2.ellipse(pupil2, ellipse, 255, thickness=-1)
            except: pass
    else:
        lid2 = iris2 = pupil2 = np.zeros((IMAGE_HEIGHT, IMAGE_WIDTH), np.uint8)
    
    # === Method3の予測 ===
    if 3 in models and models[3] is not None:
        models[3].eval()
        with torch.no_grad(), autocast():
            out3 = models[3](img_t)
            logits = out3['five_class_seg']
        
        pred = torch.argmax(logits, dim=1).cpu().numpy()[0]
        lid3 = (((pred==1) | (pred==2) | (pred==4)).astype(np.uint8)*255)
        
        # Iris: 6クラスから合成 (class 2 ∪ 3) → 楕円近似
        iris_m3_bin = (((pred==2) | (pred==3)).astype(np.uint8)*255)
        # Pupil: 6クラスから合成 (class 4 ∪ 5) → 楕円近似
        pupil_m3_bin = (((pred==4) | (pred==5)).astype(np.uint8)*255)
        
        # 楕円近似：面マスク→エッジ抽出→RANSAC→楕円マスク
        iris_m3_edge = mask_to_edge(iris_m3_bin, thickness=3)
        pupil_m3_edge = mask_to_edge(pupil_m3_bin, thickness=3)
        iris_params3  = binary_to_ellipse_params(iris_m3_edge)
        pupil_params3 = binary_to_ellipse_params(pupil_m3_edge)
        iris_mask_m3  = ellipse_params_to_mask(iris_params3,  IMAGE_HEIGHT, IMAGE_WIDTH) if iris_params3 is not None else np.zeros((IMAGE_HEIGHT,IMAGE_WIDTH), np.uint8)
        pupil_mask_m3 = ellipse_params_to_mask(pupil_params3, IMAGE_HEIGHT, IMAGE_WIDTH) if pupil_params3 is not None else np.zeros((IMAGE_HEIGHT,IMAGE_WIDTH), np.uint8)
        
        # 合成マスク（楕円近似前）と楕円マスク（楕円近似後）を重ねて表示
        iris3 = np.maximum(iris_m3_bin, iris_mask_m3)  # 両方を含める
        pupil3 = np.maximum(pupil_m3_bin, pupil_mask_m3)  # 両方を含める
    else:
        lid3 = iris3 = pupil3 = np.zeros((IMAGE_HEIGHT, IMAGE_WIDTH), np.uint8)
    
    # === Method4の予測 ===
    if 4 in models and models[4] is not None:
        models[4].eval()
        with torch.no_grad(), autocast():
            out4 = models[4](img_t)
            logits4 = out4['five_class_seg']
        
        pred4 = torch.argmax(logits4, dim=1).cpu().numpy()[0]
        lid4 = (((pred4==1) | (pred4==2) | (pred4==4)).astype(np.uint8)*255)
        
        iris_m4_bin = (((pred4==2) | (pred4==3)).astype(np.uint8)*255)
        pupil_m4_bin = (((pred4==4) | (pred4==5)).astype(np.uint8)*255)
        
        iris_m4_edge = mask_to_edge(iris_m4_bin, thickness=3)
        pupil_m4_edge = mask_to_edge(pupil_m4_bin, thickness=3)
        iris_params4  = binary_to_ellipse_params(iris_m4_edge)
        pupil_params4 = binary_to_ellipse_params(pupil_m4_edge)
        iris_mask_m4  = ellipse_params_to_mask(iris_params4,  IMAGE_HEIGHT, IMAGE_WIDTH) if iris_params4 is not None else np.zeros((IMAGE_HEIGHT,IMAGE_WIDTH), np.uint8)
        pupil_mask_m4 = ellipse_params_to_mask(pupil_params4, IMAGE_HEIGHT, IMAGE_WIDTH) if pupil_params4 is not None else np.zeros((IMAGE_HEIGHT,IMAGE_WIDTH), np.uint8)
        
        iris4 = np.maximum(iris_m4_bin, iris_mask_m4)
        pupil4 = np.maximum(pupil_m4_bin, pupil_mask_m4)
    else:
        lid4 = iris4 = pupil4 = np.zeros((IMAGE_HEIGHT, IMAGE_WIDTH), np.uint8)
    
    # === 描画 ===
    # Row 0: Eyelid
    axes[0,0].imshow(img_vis); axes[0,0].set_title(col_titles[0]); axes[0,0].axis('off')
    axes[0,0].set_ylabel(row_labels[0], fontsize=12, fontweight='bold')
    axes[0,1].imshow(lid1, cmap='gray'); axes[0,1].set_title(col_titles[1]); axes[0,1].axis('off')
    axes[0,2].imshow(lid2, cmap='gray'); axes[0,2].set_title(col_titles[2]); axes[0,2].axis('off')
    axes[0,3].imshow(lid3, cmap='gray'); axes[0,3].set_title(col_titles[3]); axes[0,3].axis('off')
    axes[0,4].imshow(lid4, cmap='gray'); axes[0,4].set_title(col_titles[4]); axes[0,4].axis('off')
    
    # Row 1: Iris
    axes[1,0].imshow(img_vis); axes[1,0].axis('off')
    axes[1,0].set_ylabel(row_labels[1], fontsize=12, fontweight='bold')
    axes[1,1].imshow(iris1, cmap='gray'); axes[1,1].axis('off')
    axes[1,2].imshow(iris2, cmap='gray'); axes[1,2].axis('off')
    axes[1,3].imshow(iris3, cmap='gray'); axes[1,3].axis('off')
    axes[1,4].imshow(iris4, cmap='gray'); axes[1,4].axis('off')
    
    # Row 2: Pupil
    axes[2,0].imshow(img_vis); axes[2,0].axis('off')
    axes[2,0].set_ylabel(row_labels[2], fontsize=12, fontweight='bold')
    axes[2,1].imshow(pupil1, cmap='gray'); axes[2,1].axis('off')
    axes[2,2].imshow(pupil2, cmap='gray'); axes[2,2].axis('off')
    axes[2,3].imshow(pupil3, cmap='gray'); axes[2,3].axis('off')
    axes[2,4].imshow(pupil4, cmap='gray'); axes[2,4].axis('off')
    
    plt.tight_layout()
    plt.show()

# Fold 0の3つのメソッドのモデルで3サンプル可視化
print("手法比較可視化 (Fold 0のモデルを使用)")
print("=" * 60)

# モデルロード（可視化対象は Method 1-4 のみ：visualize_all_methods の対応範囲）
models = {}
VISUALIZE_METHODS = [m for m in TRAIN_METHODS if m in (1, 2, 3, 4)]
for method_id in VISUALIZE_METHODS:
    model_path = MODEL_DIR / f"method{method_id}_fold0_best.pth"
    if model_path.exists():
        if method_id == 1:
            model = UNetMethod1().to(device)
        elif method_id == 2:
            model = UNetMethod2().to(device)
        elif method_id == 3:
            model = UNetMethod3().to(device)
        elif method_id == 4:
            model = UNetMethod4().to(device)
        
        checkpoint = torch.load(model_path, map_location=device)
        model.load_state_dict(checkpoint['model'])
        model.eval()
        models[method_id] = model
    else:
        print(f"⚠️ Method{method_id} モデルが見つかりません: {model_path}")
        models[method_id] = None

if len(models) > 0:
    # Validation データセット準備
    val_indices = fold_indices['0']['val']
    val_paths = [image_paths[i] for i in val_indices]
    val_ds = EyeSegmentationDataset(
        val_paths, LABEL_SEG_DIR, LABEL_OBB_DIR,
        transform=False,
        use_ellipse_cache=False,
        use_sixcls_direct=True,
    )
    
    # ランダムに3サンプル表示
    import random
    sample_indices = random.sample(range(len(val_ds)), min(3, len(val_ds)))
    
    for idx in sample_indices:
        sample = val_ds[idx]
        print(f"\nサンプル: {sample['filename']}")
        visualize_all_methods(sample, models, device)
    
    for model in models.values():
        if model is not None:
            del model
    del val_ds
    torch.cuda.empty_cache()
    print("\n✅ 可視化完了")
else:
    print("⚠️ モデルが見つかりません")
    print("   学習セル（セル12）を実行してモデルを作成してください。")


## 14. メモリクリア（オプション）


In [ ]:
import gc

# CUDAキャッシュをクリア
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("✓ CUDA cache cleared")

# ガベージコレクション
gc.collect()
print("✓ Python garbage collection completed")

print("\n=== メモリクリア完了 ===")


In [ ]:
# === tau sensitivity sweep ===
# Set RUN_TAU_SENSITIVITY=True to run. Estimated time: ~30-60 min on GPU.
# Evaluates Methods 4 and 5 (primary mode = fullmax) at τ ∈ TAU_VALUES and
# stores per-image NSD in results/cv_tau_sensitivity_*.csv.

RUN_TAU_SENSITIVITY = True

if RUN_TAU_SENSITIVITY:
    from datetime import datetime
    import pandas as pd
    import numpy as np
    import torch
    from pathlib import Path

    from evaluation.metrics_reloaded_eval import compute_hd95_nsd

    TAU_VALUES = [0.5, 1.0, 2.0, 3.0, 5.0]
    TARGET_METHODS = [4, 5]

    if 'fold_indices' not in globals():
        with open('fold_indices.json', 'r') as f:
            fold_indices = json.load(f)
    if 'image_paths' not in globals():
        IMAGES_DIR = Path("Images/images")
        df_meta = pd.read_csv('image_metadata.csv')
        image_paths = [IMAGES_DIR / row['filename'] for _, row in df_meta.iterrows()]
    MODEL_DIR = Path("model/cv_300ep") if 'MODEL_DIR' not in globals() else MODEL_DIR

    tau_rows = []
    for fold_idx in range(NUM_FOLDS):
        val_indices = fold_indices[str(fold_idx)]['val']
        val_paths = [image_paths[i] for i in val_indices]
        val_ds = EyeSegmentationDataset(
            val_paths, LABEL_SEG_DIR, LABEL_OBB_DIR,
            transform=False, use_ellipse_cache=False, use_sixcls_direct=True,
        )
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                                num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

        for method_id in TARGET_METHODS:
            model_path = MODEL_DIR / f"method{method_id}_fold{fold_idx}_best.pth"
            if not model_path.exists():
                print(f"Missing {model_path} — skipping")
                continue
            if method_id == 4:
                model = UNetMethod4().to(device)
            else:
                model = UNetMethod5().to(device)
            checkpoint = torch.load(model_path, map_location=device)
            model.load_state_dict(checkpoint['model'])
            model.eval()

            with torch.no_grad():
                for batch in tqdm(val_loader, desc=f"τ-sweep M{method_id} fold{fold_idx}", leave=False):
                    img = batch['image'].to(device)
                    gt_lid   = batch['mask_lid'].cpu().numpy()
                    gt_iris  = batch['mask_iris'].cpu().numpy()
                    gt_pupil = batch['mask_pupil'].cpu().numpy()
                    filenames = batch.get('filename', None)

                    with autocast():
                        out = model(img)

                    if method_id == 4:
                        logits = out['five_class_seg']
                        pred_labels = torch.argmax(logits, dim=1).cpu().numpy()
                    else:
                        probs = torch.sigmoid(out['amodal_logits']).cpu().numpy()

                    for b in range(img.shape[0]):
                        filename = filenames[b] if filenames is not None else str(b)
                        subject_id = str(filename).split('-', 1)[0]

                        if method_id == 4:
                            pred = pred_labels[b]
                            lid_bin = (((pred == 1) | (pred == 2) | (pred == 4)).astype(np.uint8) * 255)
                            iris_vis = ((pred == 2).astype(np.uint8) * 255)
                            iris_occ = ((pred == 3).astype(np.uint8) * 255)
                            pupil_vis = ((pred == 4).astype(np.uint8) * 255)
                            pupil_occ = ((pred == 5).astype(np.uint8) * 255)
                            iris_mask, _ = ellipse_mask_from_fullmax_contour(iris_vis, iris_occ)
                            pupil_mask, _ = ellipse_mask_from_fullmax_contour(pupil_vis, pupil_occ)
                        else:
                            lid_bin = (probs[b, 0] >= 0.5).astype(np.uint8) * 255
                            iris_raw = (probs[b, 1] >= 0.5).astype(np.uint8) * 255
                            pupil_raw = (probs[b, 2] >= 0.5).astype(np.uint8) * 255
                            iris_mask = np.zeros_like(iris_raw)
                            ell = _fit_ellipse_from_binary(iris_raw)
                            if ell is not None:
                                (cx, cy), (w, h), angle = ell
                                cv2.ellipse(iris_mask, (int(round(cx)), int(round(cy))),
                                            (max(1, int(round(w/2))), max(1, int(round(h/2)))),
                                            float(angle), 0, 360, 255, thickness=-1)
                            pupil_mask = np.zeros_like(pupil_raw)
                            ell = _fit_ellipse_from_binary(pupil_raw)
                            if ell is not None:
                                (cx, cy), (w, h), angle = ell
                                cv2.ellipse(pupil_mask, (int(round(cx)), int(round(cy))),
                                            (max(1, int(round(w/2))), max(1, int(round(h/2)))),
                                            float(angle), 0, 360, 255, thickness=-1)

                        for tau in TAU_VALUES:
                            _, nsd_lid   = compute_hd95_nsd(lid_bin,   gt_lid[b],   tau=tau)
                            _, nsd_iris  = compute_hd95_nsd(iris_mask, gt_iris[b],  tau=tau)
                            _, nsd_pupil = compute_hd95_nsd(pupil_mask, gt_pupil[b], tau=tau)
                            tau_rows.append({
                                'method': method_id, 'fold': fold_idx, 'tau': tau,
                                'filename': str(filename), 'subject_id': str(subject_id),
                                'nsd_eyelid': nsd_lid, 'nsd_iris': nsd_iris, 'nsd_pupil': nsd_pupil,
                            })

            del model
            torch.cuda.empty_cache()

        del val_loader, val_ds
        torch.cuda.empty_cache()
        print(f"Fold {fold_idx} done, rows so far: {len(tau_rows)}")

    _ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    tau_csv = Path('results') / f"cv_tau_sensitivity_{_ts}.csv"
    pd.DataFrame(tau_rows).to_csv(tau_csv, index=False)
    print(f"\n✅ τ sensitivity CSV saved: {tau_csv}  (rows = {len(tau_rows)})")
else:
    print("Set RUN_TAU_SENSITIVITY = True and re-run this cell to execute.")
